In [ ]:
import os
PATH = "/kaggle/input/datasets/jakeadam68/blackwell-deps-cu128/blackwell_deps_pack" # Le nom de ton dataset

# on installe le fichier exact de Torch 2.10.0 cu128 en premier
# Cela évite que pip ne choisisse la version 2.11 par erreur
%pip install --no-index --find-links={PATH} torch==2.10.0+cu128 torchvision torchaudio

import torch
print(f"PyTorch installé : {torch.__version__}")
print(f"CUDA disponible : {torch.cuda.is_available()}")
print(f"GPU détecté : {torch.cuda.get_device_name(0)}")

In [ ]:
# Maintenant on installe les extensions qui dépendent de la 2.10
%pip install --no-index --find-links={PATH} torch-scatter torch-sparse torch-cluster torch-spline-conv
print("Extensions PyG installées.")

In [ ]:
# Ici on installe tout le reste (Transformers, RDKit, e3nn, etc.)
# pip verra que Torch est déjà installé et ne touchera plus à la version
%pip install --no-index --find-links={PATH} torch-geometric e3nn transformers accelerate bitsandbytes safetensors huggingface_hub fair-esm rdkit pubchempy py3Dmol biopython biotite joblib tqdm pandas numpy polars pyarrow fastparquet
print("Environnement complet opérationnel !")

In [ ]:
%%bash
echo "Début de l'installation Hors-Ligne"

# Remplacez ce chemin par le vrai chemin de votre dossier my_wheels uploadé
WHEELS_DIR="/kaggle/input/datasets/jakeadam68/blackwell-deps-cu128/offline_wheels/my_wheels"

# on installe explicitement juste ce dont on a besoin, sans casser le numpy de Kaggle
pip install freesasa openmm pdbfixer --no-index --find-links "$WHEELS_DIR" --quiet

echo "Tous les packages sont installés !"

In [ ]:
import os
import random
import numpy as np
import torch
import gc
import warnings

# on désactive les warnings 
warnings.filterwarnings('ignore')
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')
# on crée une fonction pour la reproductibilité maximale 
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # on adopte la reproductibilité stricte
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)
    
    print(f"la graine aléatoire est fixée à {seed}")
    
seed_everything(42)
# on choisit une configuration optimisée pour GPU H100
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    
    print(f"\nle GPU détecté est : {gpu_name}")
    print(f"la VRAM disponible : {vram_gb:.2f} GB")
    # on choisit une optimisations Hopper / H100
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.set_float32_matmul_precision('high')
    # on active le torch.compile pour un gain de vitesse important
    print(f"le TensorFloat-32 est actif : {torch.backends.cuda.matmul.allow_tf32}")
else:
    DEVICE = torch.device("cpu")
# on procède à un nettoyage de la mémoire
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
print("la configuration de l'environnement est terminée")

In [ ]:
!echo "=== Modèle du CPU ==="
!cat /proc/cpuinfo | grep "model name" | uniq

!echo "=== Nombre de vCPUs disponibles ==="
!nproc

## **Partie Préparation des données** 

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import HeteroData, Data, Dataset
from torch_cluster import radius_graph, knn_graph
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import GroupShuffleSplit
import pandas as pd
import numpy as np
import pickle
import os
import re
import warnings
import gc
from tqdm.auto import tqdm

# Configuration
PATH_PREFIX       = "/kaggle/input/datasets/jakeadam68/bindingdb-all-202602/"
RANDOM_SEED       = 42

# Alphabet des acides aminés pour le One-Hot Encoding
AA_CHARS  = "ACDEFGHIKLMNPQRSTVWYX"
AA_TO_IDX = {aa: i for i, aa in enumerate(AA_CHARS)}
AA_VOCAB  = len(AA_CHARS)

print("Phase 1 : Chargement et Configuration")

def get_win_params(seq_len, hgvsp_str, max_len=256):
    match = re.search(r'p\.[A-Z][a-z]{2}(\d+)|p\.[A-Z](\d+)', str(hgvsp_str))
    pos = int(match.group(1) if match.group(1) else match.group(2)) if match else seq_len // 2
    center = pos - 1
    if seq_len <= max_len: return 0, seq_len, center
    s = max(0, center - max_len // 2)
    e = s + max_len
    if e > seq_len:
        e = seq_len
        s = max(0, e - max_len)
    return int(s), int(e), int(center - s)

def encode_sequence(sequence: str, mut_idx: int, plddt: torch.Tensor, metrics_vector: torch.Tensor = None, esm_vector: torch.Tensor = None) -> torch.Tensor:
    """Version optimisée sans boucle for"""
    L = len(sequence)
    if plddt.shape[0] != L:
        raise ValueError(f"Mismatch de taille : la séquence fait {L} mais le pLDDT fait {plddt.shape[0]}. "
                         f"Vérifiez le découpage de la fenêtre dans _build_data.")
    # 1. on convertit la séquence en indices en une seule fois
    indices = torch.tensor([AA_TO_IDX.get(aa.upper(), AA_TO_IDX['X']) for aa in sequence], dtype=torch.long)
    # 2. One-hot encoding vectorisé (très rapide)
    features = F.one_hot(indices, num_classes=AA_VOCAB).float()
    # 3. Ajout du pLDDT (on ne slice plus, on utilise le tensor tel quel)
    plddt_col = plddt.unsqueeze(1)
    # 4. Ajout du flag de mutation (colonne 22)
    mut_col = torch.zeros((L, 1), dtype=torch.float32)
    if 0 <= mut_idx < L:
        mut_col[mut_idx] = 1.0

    if metrics_vector is not None:
        num_metrics = metrics_vector.shape[0]
        metrics_block = torch.zeros((L, num_metrics), dtype=torch.float32)
        if 0 <= mut_idx < L:
            metrics_block[mut_idx] = metrics_vector
    else:
        # Si aucune métrique n'est fournie (ex: début de Phase 1), on met des zéros
        metrics_block = torch.zeros((L, 6), dtype=torch.float32)


    if esm_vector is not None:
        # on s'assure que l'embedding a la bonne taille (L, 1280)
        if esm_vector.shape[0] != L:
            # on ajuste la taille pour correspondre à L (padding ou truncation)
            if esm_vector.shape[0] > L:
                esm_vector = esm_vector[:L, :]
            else:
                pad = torch.zeros((L - esm_vector.shape[0], esm_vector.shape[1]), dtype=torch.float32)
                esm_vector = torch.cat([esm_vector, pad], dim=0)
    else:
        # Si pas d'ESM-2, on met des zéros
        esm_vector = torch.zeros((L, 2560), dtype=torch.float32)
        
    return torch.cat([features, plddt_col, mut_col, metrics_block, esm_vector], dim=-1)

def compute_biophysical_edges(data, radius_prot=8.0, radius_bind=5.0, k_prot=6):
    # 1. Protein -> Protein : on fusionne Radius et kNN plus efficacement
    # on utilise torch.cat puis unique pour éviter les doublons
    edge_p_p = torch.cat([
        radius_graph(data['protein'].pos, r=radius_prot, loop=False),
        knn_graph(data['protein'].pos, k=k_prot, loop=False)
    ], dim=1).unique(dim=1)
    data['protein', 'interacts', 'protein'].edge_index = edge_p_p

    # 2. Ligand -> Ligand : Graphe complet optimisé
    num_l = data['ligand'].pos.size(0)
    adj = torch.ones((num_l, num_l), device=data['ligand'].pos.device)
    data['ligand', 'interacts', 'ligand'].edge_index = adj.nonzero().t()

    # 3. Protein <-> Ligand : Bipartite Radius optimisé
    dist_mat = torch.cdist(data['protein'].pos, data['ligand'].pos)
    edge_p_l = (dist_mat < radius_bind).nonzero().t()
    data['protein', 'binds', 'ligand'].edge_index = edge_p_l
    data['ligand', 'binds', 'protein'].edge_index = edge_p_l[:, [1, 0]]

    return data
    
def extract_aa_from_hgvsp(hgvsp_str):
    """
    Extrait le résidu sauvage (WT) et le résidu muté (MT) d'une chaîne HGVS.
    Format supporté : 'ENSP...:p.Pro29Ser' ou 'p.Pro29Ser'
    """
    hgvsp_str = str(hgvsp_str)
    match = re.search(r'p\.([A-Z][a-z]{2})(\d+)([A-Z][a-z]{2})', hgvsp_str)
    if match:
        wt_full = match.group(1)
        mt_full = match.group(3)
        aa_map = {
            'Ala':'A', 'Arg':'R', 'Asn':'N', 'Asp':'D', 'Cys':'C', 
            'Glu':'E', 'Gln':'Q', 'Gly':'G', 'His':'H', 'Ile':'I', 
            'Leu':'L', 'Lys':'K', 'Met':'M', 'Phe':'F', 'Pro':'P', 
            'Ser':'S', 'Thr':'T', 'Trp':'W', 'Tyr':'Y', 'Val':'V'
        }
        return aa_map.get(wt_full), aa_map.get(mt_full)
    return None, None

In [ ]:
print("Phase 1 & 2 : Calcul des métriques atomiques")

path_prefix = "/kaggle/input/datasets/jakeadam68/bindingdb-all-202602/"
with open(path_prefix + "esmfold_foldx_hybrid_5920.pkl", "rb") as f:
    prot_3d_db = pickle.load(f)
with open(path_prefix + "structural_impact_labels_sota.pkl", "rb") as f:
    rmsd_labels = pickle.load(f)

df = pd.read_parquet(path_prefix + "bindingdb_wildtype_mutated_fullseq.parquet")

# on utilise un dictionnaire de recherche pour éviter df[df['variant_id']==vid]
seq_lookup = df.set_index('variant_id')['mutated_sequence'].to_dict()

unique_variants = df['variant_id'].unique()
variant_metrics = {}
error_log = {}

for vid in tqdm(unique_variants):
    try:
        uid, hgvsp = vid.split('_')[0], vid.split('_')[1]
        
        # 1. Récupération MT
        if vid not in prot_3d_db['mt']: continue
        mt_data = prot_3d_db['mt'][vid]
        coords_mt = np.array(mt_data['coords']) 
        plddt_mt = np.array(mt_data['plddt'])
        
        seq_mt_full = seq_lookup.get(vid)
        if seq_mt_full is None: continue
        
        # 2. Paramètres de fenêtre 
        s, e, rel_idx = get_win_params(len(seq_mt_full), hgvsp)
        
        # on vérifie que l'index relatif est valide pour MT et WT
        if rel_idx >= len(coords_mt):
            raise IndexError(f"Index relatif {rel_idx} hors limites pour MT ({len(coords_mt)})")

        # 3. Récupération WT
        win_key = (uid, s, e)
        if win_key not in prot_3d_db['wt']: continue
        coords_wt = np.array(prot_3d_db['wt'][win_key]['coords'])
        plddt_wt = np.array(prot_3d_db['wt'][win_key]['plddt'])
        
        if rel_idx >= len(coords_wt):
            raise IndexError(f"Index relatif {rel_idx} hors limites pour WT ({len(coords_wt)})")        
     
        # 4. Géométrie et Stabilité (Utilisation systématique de rel_idx)
        l_rmsd = rmsd_labels.get(vid, {}).get('local_rmsd_15A', 0.0)

        variant_metrics[vid] = {
            'structural_rmsd_A': l_rmsd,
            'true_delta_sasa': float(mt_data.get('true_delta_sasa', 0.0)),            
            'true_delta_packing': float(mt_data.get('true_delta_packing', 0.0)),      
            'true_delta_ddg': float(mt_data.get('true_delta_ddg', 0.0)),           
            'true_delta_electro': float(mt_data.get('true_delta_electro', 0.0)),
            'true_delta_solv_hydro': float(mt_data.get('true_delta_solv_hydro', 0.0)),
            'true_delta_clash': float(mt_data.get('true_delta_clash', 0.0)),      
            'plddt_confidence': mt_data['mean_plddt'] / 100.0
        }
    except Exception as e:
        err_name = type(e).__name__
        error_log[err_name] = error_log.get(err_name, 0) + 1
        continue

# Rapport d'erreurs
print(f"\nSuccès : {len(variant_metrics):,} / {len(unique_variants):,}")
if error_log:
    print("Erreurs rencontrées :")
    for err, count in error_log.items():
        print(f"{err}: {count} variants")

# Injection
if len(variant_metrics) > 0:
    metrics_df = pd.DataFrame.from_dict(variant_metrics, orient='index').reset_index().rename(columns={'index': 'variant_id'})
    cols_to_update = [col for col in metrics_df.columns if col in df.columns and col != 'variant_id']
    if cols_to_update:
        df = df.drop(columns=cols_to_update)
    df = df.merge(metrics_df, on='variant_id', how='left')
    print("Injection réussie.")
else:
    print("Aucune métrique calculée.")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

print("Vérification des distributions physiques générées :")

features_to_plot = ['true_delta_sasa', 'true_delta_packing', 
                    'true_delta_ddg', 'true_delta_solv_hydro', 'true_delta_electro', 'true_delta_clash', 'structural_rmsd_A']

fig, axes = plt.subplots(3, 3, figsize=(20, 15))
axes = axes.flatten()

for i, col in enumerate(features_to_plot):
    if col in df.columns:
        sns.histplot(df[col].dropna(), bins=50, kde=True, ax=axes[i], color='teal')
        axes[i].set_title(f"Distribution de {col}")
        axes[i].set_ylabel("Fréquence")
    else:
        print(f"Attention : la colonne {col} est introuvable dans df.")

# 3. Masquage des cases vides
for j in range(len(features_to_plot), len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.savefig("distribution_features.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
class SOTAVariantDataset(Dataset):
    
  # Constructeur mis à jour pour accepter le partage de mémoire
 def __init__(self, dataframe, ligand_db, protein_db, esm2_bundle=None, phase=1,
              protein_offsets=None, binding_offsets=None,
              protein_edges_flat=None, binding_edges_flat=None, gene_to_id=None):
     super().__init__()
     self.df = dataframe.reset_index(drop=True)
     self.ligand_db = ligand_db
     self.protein_db = protein_db
     self.esm2_bundle = esm2_bundle
     self.phase = phase
     self.ligand_edges_ram_cache = {}
     self.gene_to_id = gene_to_id if gene_to_id is not None else {}

     # Configuration Memory-Mapped 
     # Chemin d'accès Kaggle pour le modèle attaché
     model_dir = "/kaggle/input/datasets/jakeadam68/binding-edges-flat"

     # S'ils sont passés en argument, on utilise les objets partagés (Évite la duplication RAM)
     self.protein_offsets = protein_offsets
     self.binding_offsets = binding_offsets
     self.protein_edges_flat = protein_edges_flat
     self.binding_edges_flat = binding_edges_flat
      
 def __len__(self):
     return len(self.df)
 
 def __getitem__(self, idx):
     row = self.df.iloc[idx]
     vid = row['variant_id']
     smiles = row.get('Ligand SMILES', '')
     uid = row['uniprot_id']
     try:
         return self._build_data(row, vid, smiles, uid)
     except Exception as e:
         raise RuntimeError(f"Erreur d'extraction pour {vid} : {e}")
 
 def _build_data(self, row, vid, smiles, uid):

     # 1. Récupération des données brutes (Protéine entière)
     prot = self.protein_db['mt'][vid]
     full_coords = torch.tensor(prot['coords'], dtype=torch.float32)
     full_plddt = torch.tensor(prot['plddt'], dtype=torch.float32) / 100.0
     full_seq = row['mutated_sequence']

     s, e, local_mut_idx = None, None, None

     # on essaie de lire directement s, e et idx depuis mes offsets en RAM 
     if vid in self.protein_offsets:
         meta = self.protein_offsets[vid]
         if isinstance(meta, dict):
             s = meta.get('s')
             e = meta.get('e')
             local_mut_idx = meta.get('idx')

     # si absent ou non trouvé (Fallback), on calcule à la volée via HGVSP
     if s is None or e is None or local_mut_idx is None:
         s, e, local_mut_idx = get_win_params(len(full_seq), row['mutation_hgvsp'])
        
     # 3. Découpage synchronisé 
     # on ne garde que la fenêtre pour tout : séquence, coordonnées et pLDDT
     window_seq = full_seq[s:e]
     # 3. Découpage intelligent 
     # on vérifie si la base de données contient la protéine entière ou juste la fenêtre
     # si la taille est > 256, on découpe. si elle est <= 256, on considère que c'est déjà la fenêtre.
     if full_coords.shape[0] > 256:
         # Cas 1 : Protéine entière dans la DB
         pos_p_window = full_coords[s:e]
         plddt_window = full_plddt[s:e]
     else:
         # Cas 2 : Déjà découpé en fenêtre dans la DB (le cas probable ici)
         pos_p_window = full_coords
         plddt_window = full_plddt

     # on s'assure que la taille du pLDDT correspond exactement à la séquence
     # Si la DB est un peu plus courte que 256 (ex: protéine courte), on ajuste la séquence
     L_actual = plddt_window.shape[0]
     if L_actual != len(window_seq):
         window_seq = window_seq[:L_actual]

     if self.esm2_bundle is not None and vid in self.esm2_bundle['mt'] and vid in self.esm2_bundle['wt']:
         esm_data_mt = self.esm2_bundle['mt'][vid]
         esm_data_wt = self.esm2_bundle['wt'][vid]
         
         full_esm_mt = torch.tensor(esm_data_mt['emb'], dtype=torch.float32)
         full_esm_wt = torch.tensor(esm_data_wt['emb'], dtype=torch.float32)
         
         offset = esm_data_mt['offset']
         s_rel, e_rel = max(0, s - offset), min(full_esm_mt.shape[0], e - offset)
         
         esm_window_mt = full_esm_mt[s_rel:e_rel]
         esm_window_wt = full_esm_wt[s_rel:e_rel]
         
         # 1. Calcul du Delta (L'impact de la mutation sur le langage des protéines)
         delta_esm_window = esm_window_mt - esm_window_wt
         
         # 2. Concaténation (MT + Delta) -> Dimension 2560
         esm_combined = torch.cat([esm_window_mt, delta_esm_window], dim=-1)
         
         if esm_combined.shape[0] < 256:
             padding = torch.zeros((256 - esm_combined.shape[0], 2560), dtype=torch.float32)
             esm_combined = torch.cat([esm_combined, padding], dim=0)
         else:
             esm_combined = esm_combined[:256]
     else:
         esm_combined = torch.zeros((256, 2560), dtype=torch.float32)

     metrics_cols = ['true_delta_sasa', 'true_delta_packing', 'true_delta_solv_hydro', 'true_delta_electro', 'true_delta_clash', 'true_delta_ddg']

     if self.phase == 1:
         # En Phase 1 : Aucun Ligand. on construit un graphe homogène.
         data = Data()
         x_local = encode_sequence(window_seq, local_mut_idx, plddt_window, metrics_vector=torch.zeros(6), esm_vector=esm_combined)
         
         data.x = x_local
         data.pos = pos_p_window - pos_p_window[local_mut_idx].clone() # Centrage
         data.local_mut_idx = torch.tensor([local_mut_idx], dtype=torch.long)
         data.weight = torch.tensor([row['informational_weight']], dtype=torch.float32) # Poids info
         
         # Arrêtes protéine-protéine
         if vid in self.protein_offsets and self.protein_edges_flat is not None:
             meta = self.protein_offsets[vid]
             start, end = (meta['start'], meta['end']) if isinstance(meta, dict) else meta
             if self.protein_edges_flat.ndim == 2 and self.protein_edges_flat.size(0) == 2:
                 data.edge_index = self.protein_edges_flat[:, start:end]
             else:
                 data.edge_index = self.protein_edges_flat[start:end]

         else:
             # Fallback propre pour la Phase 1
             data.edge_index = torch.cat([
                 radius_graph(data.pos, r=8.0, loop=False),
                 knn_graph(data.pos, k=6, loop=False)
             ], dim=1).unique(dim=1)
         

         # Cibles (Targets) pour la Phase 1 : RMSD Class + Les 7 régressions
         #data.y_rmsd_class = torch.tensor([row['l_rmsd_class']], dtype=torch.long)
         
         # on groupe les 7 deltas dans un seul tenseur pour la Multitask Loss
         #deltas = [row['delta_plddt']] + [row.get(col, 0.0) for col in metrics_cols]
         #data.y_deltas = torch.tensor([deltas], dtype=torch.float32) # Shape [1, 7]
         targets = [row['structural_rmsd_A']] + [row.get(col, 0.0) for col in metrics_cols]
         data.y_deltas = torch.tensor([targets], dtype=torch.float32) # Shape [1, 7]
         # Création du masque de perte (Loss Masking)
         # Indique si FoldX a planté (ex: généré via une colonne 'foldx_valid' définie avant le Dataset)
         foldx_valid = 1.0 if row.get('foldx_valid', True) else 0.0
            
         # Indices : [RMSD, SASA, Packing, Solv, Electro, Clash, DDG]
         # RMSD, SASA et Packing sont toujours valides (1.0). FoldX dépend du flag.
         data.target_mask = torch.tensor([[1.0, 1.0, 1.0, foldx_valid, foldx_valid, foldx_valid, foldx_valid]], dtype=torch.float32)

         
         return data


     else:
         # Phase 2 : on injecte les valeurs comme features
         data = HeteroData()
         # Extraire les valeurs réelles du dataframe
         metrics_values = torch.tensor([row.get(col, 0.0) for col in metrics_cols], dtype=torch.float32)
         x_local = encode_sequence(window_seq, local_mut_idx, plddt_window, metrics_vector=metrics_values, esm_vector=esm_combined)

         data['protein'].x = x_local
         pivot = pos_p_window[local_mut_idx].clone()
         data['protein'].pos = pos_p_window - pivot
         data.local_mut_idx = torch.tensor([local_mut_idx], dtype=torch.long)
         
         if smiles in self.ligand_db:
             # Géométrie et Features Ligands 
             lig = self.ligand_db[smiles]
             data['ligand'].x = torch.tensor(lig['atom_features'], dtype=torch.float32)
             data['ligand'].pos = torch.tensor(lig['coords'], dtype=torch.float32) - torch.tensor(lig['coords'], dtype=torch.float32).mean(dim=0)
             data.ligand_global_feat = torch.tensor(lig['global_rdkit'], dtype=torch.float32)
         else:
                raise ValueError(f"SMILES introuvable dans la base de données : {smiles}")


         # 1. Ligand-Ligand
         if smiles not in self.ligand_edges_ram_cache:
             # on génère le graphe complet une seule fois pour ce SMILES
             num_l = data['ligand'].pos.size(0)
             adj = torch.ones((num_l, num_l))
             adj.fill_diagonal_(0)
             self.ligand_edges_ram_cache[smiles] = adj.nonzero().t()

         # on récupère l'index depuis la RAM 
         data['ligand', 'interacts', 'ligand'].edge_index = self.ligand_edges_ram_cache[smiles]

         # 1. Protéine-Protéine (Slicing du tenseur plat mappé)
         if vid in self.protein_offsets and self.protein_edges_flat is not None:
             meta = self.protein_offsets[vid]
             # Gestion de la rétrocompatibilité (si vos anciens offsets étaient de simples tuples)
             start, end = (meta['start'], meta['end']) if isinstance(meta, dict) else meta
            
             # Découpage dynamique de l'intervalle
             if self.protein_edges_flat.ndim == 2 and self.protein_edges_flat.size(0) == 2:
                 data['protein', 'interacts', 'protein'].edge_index = self.protein_edges_flat[:, start:end]
             else:
                 data['protein', 'interacts', 'protein'].edge_index = self.protein_edges_flat[start:end]
         else:
             # Fallback si manque dans le cache
             data = compute_biophysical_edges(data)
    
         # 3. Protéine-Ligand (Slicing du tenseur plat mappé)
         pair_key = (vid, smiles)
         if pair_key in self.binding_offsets and self.binding_edges_flat is not None:
             meta = self.binding_offsets[pair_key]
             if isinstance(meta, dict):
                 start, end = meta['start'], meta['end']
             else:
                 start, end = meta
            
             # Slicing robuste selon les dimensions réelles des arêtes
             if self.binding_edges_flat.ndim == 2 and self.binding_edges_flat.size(0) == 2:
                 edge_index = self.binding_edges_flat[:, start:end]
                 data['protein', 'binds', 'ligand'].edge_index = edge_index
                 # Pour inverser au format [2, N], on permute les lignes 0 et 1
                 data['ligand', 'binds', 'protein'].edge_index = edge_index[[1, 0], :]
             else:
                 edge_index = self.binding_edges_flat[start:end]
                 data['protein', 'binds', 'ligand'].edge_index = edge_index
                 # Pour inverser au format [N, 2], on permute les colonnes 0 et 1
                 data['ligand', 'binds', 'protein'].edge_index = edge_index[:, [1, 0]]
         else:
             # Calcul on-the-fly si manquant
             dist_mat = torch.cdist(data['protein'].pos, data['ligand'].pos)
             edge_p_l = (dist_mat < 5.0).nonzero().t()
             data['protein', 'binds', 'ligand'].edge_index = edge_p_l
             # Comme nonzero().t() renvoie un tenseur de forme [2, N], l'inversion permute les lignes 0 et 1
             data['ligand', 'binds', 'protein'].edge_index = edge_p_l[[1, 0], :]

        
         # Cibles d'affinité (Phase 2)
         data.y_delta_pAff = torch.tensor([row['delta_pAff']], dtype=torch.float32)
         data.weight = torch.tensor([row['label_weight']], dtype=torch.float32)

         # Conversion du gène en entier pour la pairwise ranking loss
         g_id = self.gene_to_id.get(row['gene_symbol'], 0)
         data.gene_id = torch.tensor([g_id], dtype=torch.long) 

         return data

In [ ]:
from sklearn.model_selection import GroupShuffleSplit
import pandas as pd
import numpy as np

print("Phase Split Silver : Généralisation en utilisant une stratégie stratified Group Split sur les données du dataset général")

# 1. Calcul de la popularité des gènes
gene_counts = df['gene_symbol'].value_counts()
# on crée 3 catégories de popularité : Rare, Moyen, Fréquent
# on utilise les quantiles pour que les groupes soient équilibrés en nombre de gènes
low = gene_counts.quantile(0.33)
high = gene_counts.quantile(0.66)

def assign_strat(count):
    if count <= low: return 'rare'
    elif count <= high: return 'medium'
    else: return 'frequent'

# on crée un mapping gène -> strate
gene_strat_map = gene_counts.apply(assign_strat).to_dict()
df['gene_strat'] = df['gene_symbol'].map(gene_strat_map)

# 2. Split stratifié par groupe
# on fait le split pour chaque strate séparément pour garantir l'équilibre
df_train, df_val, df_test = pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

for strat in ['rare', 'medium', 'frequent']:
    df_strat = df[df['gene_strat'] == strat].copy()
    
    # Split 1: Train vs (Val + Test)
    gss1 = GroupShuffleSplit(n_splits=1, train_size=0.7, random_state=RANDOM_SEED)
    train_idx, rem_idx = next(gss1.split(df_strat, groups=df_strat['gene_symbol']))
    
    df_train_s = df_strat.iloc[train_idx]
    df_rem_s = df_strat.iloc[rem_idx]
    
    # Split 2: Val vs Test (50/50 du reste)
    gss2 = GroupShuffleSplit(n_splits=1, train_size=0.5, random_state=RANDOM_SEED)
    val_idx, test_idx = next(gss2.split(df_rem_s, groups=df_rem_s['gene_symbol']))
    
    df_val_s = df_rem_s.iloc[val_idx]
    df_test_s = df_rem_s.iloc[test_idx]
    
    # Accumulation
    df_train = pd.concat([df_train, df_train_s])
    df_val = pd.concat([df_val, df_val_s])
    df_test = pd.concat([df_test, df_test_s])

# 3. Vérification finale
genes_train = set(df_train['gene_symbol'].unique())
genes_val = set(df_val['gene_symbol'].unique())
genes_test = set(df_test['gene_symbol'].unique())

assert genes_train.isdisjoint(genes_val), "Fuite Train/Val !"
assert genes_train.isdisjoint(genes_test), "Fuite Train/Test !"
assert genes_val.isdisjoint(genes_test), "Fuite Val/Test !"

total_pairs = len(df)
print(f"Répartition équilibrée et stratifiée (Total: {total_pairs:,} paires)")

# Affichage détaillé pour chaque set
print(f"Train set : {len(df_train)} paires | {len(genes_train)} gènes | {len(df_train)/total_pairs:.2%} du total")
print(f"Val set   : {len(df_val):} paires | {len(genes_val)} gènes | {len(df_val)/total_pairs:.2%} du total")
print(f"Test set  : {len(df_test):} paires | {len(genes_test)} gènes | {len(df_test)/total_pairs:.2%} du total")


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.cluster import KMeans

print("Phase Split gold : Stratified Group Split Indépendant du fichier globale pour la phase 2")

# 1. Chargement du vrai Gold Standard de 16k lignes (strictement expérimental)
df_gold_raw = pd.read_parquet("/kaggle/input/datasets/jakeadam68/bindingdb-all-202602/gold_standard_experimental_16k.parquet")

# =============================================================================
# 2. Group Split Indépendant et équilibré sur le gold standard (70/15/15)
# =============================================================================
# Calcul de la popularité des gènes uniquement sur les paires Gold
gold_gene_counts = df_gold_raw['gene_symbol'].value_counts()
low = gold_gene_counts.quantile(0.33)
high = gold_gene_counts.quantile(0.66)

def assign_gold_strat(count):
    if count <= low: return 'rare'
    elif count <= high: return 'medium'
    else: return 'frequent'

gold_gene_strat_map = gold_gene_counts.apply(assign_gold_strat).to_dict()
df_gold_raw['gene_strat'] = df_gold_raw['gene_symbol'].map(gold_gene_strat_map)

# Split stratifié par gène pour un équilibre parfait de l'évaluation
df_gold_train, df_gold_val, df_gold_test = pd.DataFrame(), pd.DataFrame(), pd.DataFrame()
RANDOM_SEED = 42

for strat in ['rare', 'medium', 'frequent']:
    df_strat = df_gold_raw[df_gold_raw['gene_strat'] == strat].copy()
    
    # Split 1: Train vs (Val + Test) - 70% Train, 30% Reste
    gss1 = GroupShuffleSplit(n_splits=1, train_size=0.7, random_state=RANDOM_SEED)
    train_idx, rem_idx = next(gss1.split(df_strat, groups=df_strat['gene_symbol']))
    
    df_train_s = df_strat.iloc[train_idx]
    df_rem_s = df_strat.iloc[rem_idx]
    
    # Split 2: Val vs Test (50/50 du reste, donc 15% / 15%)
    gss2 = GroupShuffleSplit(n_splits=1, train_size=0.5, random_state=RANDOM_SEED)
    val_idx, test_idx = next(gss2.split(df_rem_s, groups=df_rem_s['gene_symbol']))
    
    df_val_s = df_rem_s.iloc[val_idx]
    df_test_s = df_rem_s.iloc[test_idx]
    
    df_gold_train = pd.concat([df_gold_train, df_train_s])
    df_gold_val = pd.concat([df_gold_val, df_val_s])
    df_gold_test = pd.concat([df_gold_test, df_test_s])

# Vérification d'absence de fuite génétique
genes_gold_train = set(df_gold_train['gene_symbol'].unique())
genes_gold_val = set(df_gold_val['gene_symbol'].unique())
genes_gold_test = set(df_gold_test['gene_symbol'].unique())

assert genes_gold_train.isdisjoint(genes_gold_val), "Fuite Train/Val dans le Gold !"
assert genes_gold_train.isdisjoint(genes_gold_test), "Fuite Train/Test dans le Gold !"
assert genes_gold_val.isdisjoint(genes_gold_test), "Fuite Val/Test dans le Gold !"

total_gold = len(df_gold_raw)
print(f"Répartition équilibrée et stratifiée (Total: {total_gold:,} paires)")

print(f"Train Gold : {len(df_gold_train):,} paires | {len(genes_gold_train)} gènes | {len(df_gold_train)/total_gold:.2%}")
print(f"Val Gold   : {len(df_gold_val):,} paires | {len(genes_gold_val)} gènes | {len(df_gold_val)/total_gold:.2%}")
print(f"Test Gold  : {len(df_gold_test):,} paires | {len(genes_gold_test)} gènes | {len(df_gold_test)/total_gold:.2%}")


# Nettoyage des colonnes temporaires
df_gold_train.drop(columns=['gene_strat'], errors='ignore', inplace=True)
df_gold_val.drop(columns=['gene_strat'], errors='ignore', inplace=True)
df_gold_test.drop(columns=['gene_strat'], errors='ignore', inplace=True)

# 4. Calcul des quantiles RMSD 
rmsd_train_unique = df_train.drop_duplicates(subset=['variant_id'])['structural_rmsd_A'].dropna()
q33 = rmsd_train_unique.quantile(0.33333)
q66 = rmsd_train_unique.quantile(0.66667)


print("\nPhase 3 exécutée avec succès.")

In [ ]:
from sklearn.preprocessing import QuantileTransformer
import pickle

print("Correction du biais de gène")

# 1. Injection de la physique dans le gold standard 

print("Injection des métriques physiques dans le fichier expérimentale qui servira pour le fine-tunning...")

# on supprime delta_pAff de metrics_df au cas où, pour éviter les doublons
metrics_df_clean = metrics_df.drop(columns=['delta_pAff'], errors='ignore')

# Les 7 colonnes physiques cibles d'AEGIS-GT
cols_physics = [
    'structural_rmsd_A', 'true_delta_sasa', 'true_delta_packing', 
    'true_delta_solv_hydro', 'true_delta_electro', 'true_delta_clash', 'true_delta_ddg'
]

# on supprime toutes les colonnes physiques et les résidus de fusions précédents (_x, _y)
# pour éviter tout conflit de merge lors de la ré-exécution de la cellule.
print("Nettoyage des colonnes physiques et résidus de fusions...")

for name, dset in zip(['df_train', 'df_val', 'df_test', 'df_gold_train', 'df_gold_val', 'df_gold_test'], 
                      [df_train, df_val, df_test, df_gold_train, df_gold_val, df_gold_test]):
    
    # on repère toutes les colonnes de physique ou polluées par des suffixes _x/_y
    cols_to_drop = [
        col for col in dset.columns 
        if col in cols_physics or col == 'plddt_confidence' or col.endswith('_x') or col.endswith('_y')
    ]
    if cols_to_drop:
        dset.drop(columns=cols_to_drop, errors='ignore', inplace=True)

print("Injection propre des métriques physiques...")
df_train = df_train.merge(metrics_df_clean, on='variant_id', how='left')
df_val = df_val.merge(metrics_df_clean, on='variant_id', how='left')
df_test = df_test.merge(metrics_df_clean, on='variant_id', how='left')

# on fusionne la table metrics_df sur le variant_id pour apporter 'plddt_confidence' et la physique
df_gold_train = df_gold_train.merge(metrics_df_clean, on='variant_id', how='left')
df_gold_val   = df_gold_val.merge(metrics_df_clean, on='variant_id', how='left')
df_gold_test  = df_gold_test.merge(metrics_df_clean, on='variant_id', how='left')

# on détecte les plantages de FoldX (quand ddg et solv_hydro valent exactement 0.0)
# on applique ça sur les jeux Silver (Phase 1) et Gold (Phase 2)
for dset in [df_train, df_val, df_test, df_gold_train, df_gold_val, df_gold_test]:
    dset['foldx_valid'] = ~((dset['true_delta_ddg'] == 0.0) & (dset['true_delta_solv_hydro'] == 0.0))

# 1. Paramètres de l'Information Effective
beta = 0.9999  # plus beta est proche de 1, plus on s'approche de l'ICF classique
epsilon = 1e-6

# 2. Calcul des fréquences
unique_variants_df = df.drop_duplicates(subset=['variant_id'])
gene_counts = unique_variants_df.groupby('gene_symbol').size()

global_std = df_gold_train['delta_pAff'].std()
print(f"Écart-type réel de delta_pAff détecté : {global_std:.4f}")

# 3. Calcul du Nombre Effectif (Effective Number of Samples)
# Formule : En = (1 - beta^n) / (1 - beta)
# on intègre la diversité (global_std) pour pondérer l'importance
eff_num = (1.0 - np.power(beta, gene_counts)) / (1.0 - beta)

# on pondère l'information effective par la diversité structurelle
# on utilise l'inverse car plus l'information effective est grande, plus le poids doit être bas
gene_weights = 1.0 / (eff_num * global_std + epsilon)

# 4. Normalisation Rigoureuse (Moyenne = 1.0)
# c'est l'étape la plus importante pour la stabilité du gradient
gene_weights = gene_weights / gene_weights.mean()

# Stabilisation du Gradient
# Même avec l'information effective, un ratio de 700k est trop dangereux.
# on applique un "Smooth Capping" : on limite le poids max à 100x la moyenne.
# cela ne détruit pas l'information, mais empêche un seul exemple de faire exploser le gradient.
max_weight = 100.0 * gene_weights.mean()

gene_weights = np.clip(gene_weights, a_min=None, a_max=max_weight)

# on renormalise une dernière fois après le clip
gene_weights = gene_weights / gene_weights.mean()

# Mappage sur l'intégralité du dataset
df_train['informational_weight'] = df_train['gene_symbol'].map(gene_weights).astype(np.float32)
df_val['informational_weight'] = df_val['gene_symbol'].map(gene_weights).astype(np.float32)
df_test['informational_weight'] = df_test['gene_symbol'].map(gene_weights).astype(np.float32)

# Mappage des poids d'information sur les 3 Gold sets réels
df_gold_train['informational_weight'] = df_gold_train['gene_symbol'].map(gene_weights).astype(np.float32)
df_gold_val['informational_weight']   = df_gold_val['gene_symbol'].map(gene_weights).astype(np.float32)
df_gold_test['informational_weight']  = df_gold_test['gene_symbol'].map(gene_weights).astype(np.float32)

# 2. Poids de la Source et de la Structure 
source_map = {'Ki (nM)': 1.0, 'Kd (nM)': 1.0, 'IC50 (nM)': 0.5, 'EC50 (nM)': 0.2}

def structural_confidence(plddt):
    return 1.0 / (1.0 + np.exp(-20 * (plddt - 0.7)))

for target_df in [df_gold_train, df_gold_val, df_gold_test]:
    # 1. Poids Expérimental
    target_df['source_weight'] = target_df['aff_type'].map(source_map).fillna(0.2)
    # 2. Poids Structurel (ESMFold)
    target_df['struct_weight'] = structural_confidence(target_df['plddt_confidence'].fillna(0))
    target_df['label_weight'] = (target_df['informational_weight'] * target_df['source_weight'] * target_df['struct_weight']).astype(np.float32)

# 2. on utilise uniquement le QuantileTransformer 
scaler = QuantileTransformer(output_distribution='normal', n_quantiles=300, random_state=42)

train_unique_for_scaler = df_train.drop_duplicates(subset=['variant_id'])
scaler.fit(train_unique_for_scaler[cols_physics].fillna(0))

# 3. Application
df_train[cols_physics] = scaler.transform(df_train[cols_physics].fillna(0))
df_val[cols_physics]   = scaler.transform(df_val[cols_physics].fillna(0))
df_test[cols_physics]  = scaler.transform(df_test[cols_physics].fillna(0))

# Normalisation du Gold Standard réel
df_gold_train[cols_physics] = scaler.transform(df_gold_train[cols_physics].fillna(0))
df_gold_val[cols_physics]   = scaler.transform(df_gold_val[cols_physics].fillna(0))
df_gold_test[cols_physics]  = scaler.transform(df_gold_test[cols_physics].fillna(0))

# Sauvegarde du scaler pour inverser les prédictions plus tard
with open("phase1_scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

# Sauvegarde des fichiers gold sets réels normalisés
df_gold_train.to_parquet("df_gold_train_clean.parquet", index=False)
df_gold_val.to_parquet("df_gold_val_clean.parquet", index=False)
df_gold_test.to_parquet("df_gold_test_clean.parquet", index=False)

print("Normalisation Quantile globale et Pondérations appliquées avec succès.")

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd # Ajout de l'import pour être sûr
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

def audit_feature_redundancy(df, features, targets=None):
    # 1. Vérification des colonnes
    all_cols = features + (targets if targets else [])
    missing_cols = [col for col in all_cols if col not in df.columns]
    if missing_cols:
        print(f"Les colonnes suivantes sont manquantes : {missing_cols}")
        return 
    
    print("Audit de Redondance des Métriques Biophysiques")

    # on supprime les NaN pour ne pas fausser les corrélations et le VIF
    df_clean = df[all_cols].dropna()
    
    # A. Matrice de corrélation (Features + Targets)
    # Ici, on peut garder les targets pour voir si les features sont liées à la cible

    plt.figure(figsize=(12, 10))
    corr_matrix = df_clean.corr()
    sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f")
    plt.title("Matrice de Corrélation : Features Biophysiques vs Cibles")
    plt.savefig("correlation_matrix_full.png", dpi=300, bbox_inches='tight')
    plt.show()
    
    # B. Calcul du Variance Indicator Factor (Uniquement sur les features)
    
    print("\n📈 Calcul du Variance Inflation Factor (VIF) sur les Features...")
    X = df_clean[features]
    X_vif = add_constant(X)
    
    vif_data = pd.DataFrame()
    vif_data["feature"] = X.columns
    vif_data["VIF"] = [variance_inflation_factor(X_vif.values, i+1) for i in range(len(X.columns))]
    
    print(vif_data.sort_values(by="VIF", ascending=False))
    print("\nInterprétation : VIF < 5 : Faible | 5 < VIF < 10 : Modéré | VIF > 10 : Forte multicolinéarité")

# Application

# 1. on définit uniquement les variables d'entrée 
features_to_audit = [
    'true_delta_sasa', 'true_delta_packing', 'true_delta_ddg', 
    'true_delta_solv_hydro', 'true_delta_electro', 'true_delta_clash',
    'structural_rmsd_A'
]

# 2. on définit les cibles pour la visualisation de la corrélation
targets_to_audit = [
    'delta_pAff'
]

audit_feature_redundancy(df_gold_train, features_to_audit, targets_to_audit)

In [ ]:
import pandas as pd
import numpy as np

print("Audit des Statistiques Post-Normalisation Quantile Transformer")

cols_to_scale = ['structural_rmsd_A', 'true_delta_sasa', 'true_delta_packing', 'true_delta_ddg', 
                 'true_delta_solv_hydro', 'true_delta_electro', 'true_delta_clash']

audit_stats = []

# on extrait les variants uniques du train set pour vérifier le scaler pur
df_train_unique = df_train.drop_duplicates(subset=['variant_id'])

for col in cols_to_scale:
    audit_stats.append({
        'Feature': col,
        'Train Unique Mean': df_train_unique[col].mean(), # La vraie moyenne sans biais de ligand
        'Train Unique Std': df_train_unique[col].std(),
        'Train Full Mean': df_train[col].mean(),          # La moyenne pondérée par les ligands
        'Train Full Std': df_train[col].std()
    })

audit_df = pd.DataFrame(audit_stats).set_index('Feature')

# on affiche le tableau arrondi à 4 décimales pour la lisibilité
display(audit_df.round(4))

# Vérification algorithmique sur le Train Set
train_mean_ok = np.allclose(audit_df['Train Unique Mean'], 0, atol=0.15)
train_std_ok = np.allclose(audit_df['Train Unique Std'], 1, atol=0.15) # L'écart-type pandas vs sklearn peut avoir une micro-différence due au degré de liberté ( ddof = 1 pour pandas & ddof = 0 pour scikit learn)


if train_mean_ok and train_std_ok:
    print("\nSuccès absolu : L'espace latent est parfaitement conditionné.")
    print("Moyennes proches de 0 et écarts-types proches de 1. SwiGLU et la Huber Loss vont converger sans problème !")
else:
    print("\nLes données semblent s'écarter de la loi normale.")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

def audit_gene_weights(df):
    print("Audit des Poids d'Importance par Gène")
    
    # 1. on crée un DataFrame résumé par gène
    # Mon utilise 'nunique' pour compter la biologie (variations), pas la chimie
    gene_analysis = df.groupby('gene_symbol').agg({
        'informational_weight': 'mean',
        'variant_id': 'nunique'  
    }).rename(columns={'variant_id': 'unique_variants'})
    
    # 2. Statistiques descriptives
    print("\nStatistiques des Poids")
    print(f"Poids Minimum : {gene_analysis['informational_weight'].min():.6f}")
    print(f"Poids Maximum : {gene_analysis['informational_weight'].max():.6f}")
    print(f"Poids Moyen   : {gene_analysis['informational_weight'].mean():.6f}")
    print(f"Ratio Max/Min  : {gene_analysis['informational_weight'].max() / gene_analysis['informational_weight'].min():.2f}x")
    
    # 3. Identification des gènes extrêmes
    print("\nTop 5 Gènes les plus diversifiés (Poids Faible)")
    print(gene_analysis.sort_values('informational_weight').head(5))
    print("\nTop 5 Gènes les plus rares (Poids Fort)")
    print(gene_analysis.sort_values('informational_weight', ascending=False).head(5))
    
    # 4. Visualisation : Corrélation Fréquence vs Poids
    plt.figure(figsize=(10, 6))
    sns.scatterplot(data=gene_analysis, x='unique_variants', y='informational_weight', alpha=0.7, color='purple')
    plt.xscale('log') # Échelle logarithmique
    plt.yscale('log') # Échelle logarithmique
    plt.title("Relation entre la Diversité du Gène et son Poids d'Importance")
    plt.xlabel("Nombre de variants uniques (échelle log)")
    plt.ylabel("Poids attribué (échelle log)")
    plt.grid(True, which="both", ls="-", alpha=0.2)
    plt.savefig("distribution_poids_par_gène.png", dpi=300, bbox_inches='tight')
    plt.show()
    
    return gene_analysis

# Application sur le set d'entraînement
gene_stats = audit_gene_weights(df_train)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

print("Visualisation des distributions post normalisation :")

features_to_plot = [
    'true_delta_sasa', 'true_delta_packing', 
    'true_delta_solv_hydro', 'true_delta_electro', 
    'true_delta_clash', 'true_delta_ddg', 'structural_rmsd_A'
]

# Grille 3x3 pour nos 9 variables
fig, axes = plt.subplots(3, 3, figsize=(20, 15))
axes = axes.flatten()

# on isole les variants uniques pour voir la vraie loi normale sans le biais des ligands
df_train_unique = df_train.drop_duplicates(subset=['variant_id'])

for i, col in enumerate(features_to_plot):
    
    sns.histplot(df_train_unique[col].dropna(), bins=50, kde=True, ax=axes[i], color='indigo')
    
    # Ajout d'une ligne verticale rouge pour marquer le zéro parfait
    axes[i].axvline(0, color='red', linestyle='--', alpha=0.5)
    
    axes[i].set_title(f"Post-Norm: {col}")
    axes[i].set_ylabel("Fréquence")

# Masquage des cases vides (indices 7 et 8)
for j in range(len(features_to_plot), len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.savefig("distribution_features_normalized.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import torch
from torch.utils.data import WeightedRandomSampler
import pickle


print("Création des Datasets PyTorch")

# Chargement direct des gold sets réels, normalisés et nettoyés
df_gold_train_clean = pd.read_parquet("df_gold_train_clean.parquet")
df_gold_val_clean   = pd.read_parquet("df_gold_val_clean.parquet")
df_gold_test_clean  = pd.read_parquet("df_gold_test_clean.parquet")

# 2. Chargement de la base de données géométrique des ligands (RDKit) [2.1] nécessaire uniquement en Phase 2 pour construire les graphes moléculaires
print("Chargement de la base de données géométrique des ligands...")
try:
    with open(path_prefix + "ligands_3d_rdkit_database.pkl", "rb") as f:
        ligand_db = pickle.load(f)
    print("Base de données des ligands chargée avec succès.")
except Exception as e:
    print(f"Erreur lors du chargement des ligands : {e}")
    ligand_db = {}

def filter_valid_samples(df, prot_3d_db, ligand_db=None, phase=1):
    """
    Filtre vectoriel intelligent selon la phase.
    Phase 1 : Vérifie uniquement la présence de la protéine.
    Phase 2 : Vérifie Protéine + Ligand.
    """
    valid_proteins = set(prot_3d_db['mt'].keys())
    initial_len = len(df)
    
    # Masque de base (Protéines)
    mask = df['variant_id'].isin(valid_proteins)
    
    # Masque additionnel (Ligands) uniquement si Phase 2
    if phase == 2 and ligand_db is not None:
        valid_ligands = set(ligand_db.keys())
        mask = mask & df['Ligand SMILES'].isin(valid_ligands)
        
    filtered_df = df[mask].copy()
    print(f"Filtrage (Phase {phase}) : {len(filtered_df):,} / {initial_len:,} paires valides conservées "
          f"({len(filtered_df)/initial_len*100:.2f}%)")
    return filtered_df


esm_path = "/kaggle/input/datasets/jakeadam68/bindingdb-all-202602/esm2_pathogenic_bundle_local.pkl"

try:
    with open(esm_path, "rb") as f:
        esm2_bundle = pickle.load(f)
    print("ESM-2 Bundle chargé avec succès.")
except FileNotFoundError:
    print(f"Erreur : Le fichier {esm_path} est introuvable. Vérifiez le chemin !")
    esm2_bundle = None
except Exception as e:
    print(f"Erreur lors du chargement de ESM-2 : {e}")
    esm2_bundle = None

# Chargement unique et partagé des offsets et tenseurs (
print("\nChargement unique des offsets et mappage des fichiers plats...")
model_dir = "/kaggle/input/datasets/jakeadam68/binding-edges-flat"

shared_protein_offsets = torch.load(f"{model_dir}/protein_offsets.pt", map_location='cpu')
shared_binding_offsets = torch.load(f"{model_dir}/binding_offsets.pt", map_location='cpu')
shared_protein_edges_flat = torch.load(f"{model_dir}/protein_edges_flat.pt",map_location='cpu', mmap=True)
shared_binding_edges_flat = torch.load(f"{model_dir}/binding_edges_flat.pt", map_location='cpu', mmap=True)

print("Création du dictionnaire des gènes...")
unique_genes = df_gold_train_clean['gene_symbol'].unique()
global_gene_to_id = {gene: idx for idx, gene in enumerate(unique_genes)}

# ==========================================
# 4. Datasets pour Phase 2 (Phase 2 : Apprentissage par transfert inter-modal et ajustement fin partiel pour la prédiction d'affinité.)
# ==========================================
print("\nPréparation des Datasets pour la Phase 2...")

df_gold_train_clean = filter_valid_samples(df_gold_train_clean, prot_3d_db, ligand_db, phase=2)
df_gold_val_clean   = filter_valid_samples(df_gold_val_clean, prot_3d_db, ligand_db, phase=2)
df_gold_test_clean  = filter_valid_samples(df_gold_test_clean, prot_3d_db, ligand_db, phase=2)

gold_train_dataset = SOTAVariantDataset(
    df_gold_train_clean, ligand_db, prot_3d_db, esm2_bundle, phase=2,
    protein_offsets=shared_protein_offsets,
    binding_offsets=shared_binding_offsets,
    protein_edges_flat=shared_protein_edges_flat,
    binding_edges_flat=shared_binding_edges_flat,
    gene_to_id=global_gene_to_id
)

gold_val_dataset = SOTAVariantDataset(
    df_gold_val_clean, ligand_db, prot_3d_db, esm2_bundle, phase=2,
    protein_offsets=shared_protein_offsets,
    binding_offsets=shared_binding_offsets,
    protein_edges_flat=shared_protein_edges_flat,
    binding_edges_flat=shared_binding_edges_flat,
    gene_to_id=global_gene_to_id
)

gold_test_dataset = SOTAVariantDataset(
    df_gold_test_clean, ligand_db, prot_3d_db, esm2_bundle, phase=2,
    protein_offsets=shared_protein_offsets,
    binding_offsets=shared_binding_offsets,
    protein_edges_flat=shared_protein_edges_flat,
    binding_edges_flat=shared_binding_edges_flat,
    gene_to_id=global_gene_to_id
)

print(f"\nDatasets PyTorch générés avec succès :")
print(f"➜ Phase 2 (Gold Train) : {len(gold_train_dataset):,} paires réelles")
print(f"➜ Phase 2 (Gold Val)   : {len(gold_val_dataset):,} paires réelles")
print(f"➜ Phase 2 (Gold Test)  : {len(gold_test_dataset):,} paires réelles")

## **Partie Architecture du modèle Transformer Géométrique**

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_scatter import scatter_mean, scatter_add, scatter_softmax
from torch_geometric.nn import radius_graph
from torch_geometric.utils import to_dense_batch, to_dense_adj
from torch.utils.checkpoint import checkpoint

# =============================================================================
# 1. RMSNORM & SWIGLU
# =============================================================================
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps, self.weight = eps, nn.Parameter(torch.ones(dim))
    def forward(self, x):
        # on sauvegarde le type d'origine (ex: float16 ou bfloat16)
        orig_dtype = x.dtype
        # on passe temporairement en float32 pour sécuriser le calcul carrés et la moyenne
        x_fp32 = x.to(torch.float32)
        variance = x_fp32.pow(2).mean(-1, keepdim=True)
        x_norm = x_fp32 * torch.rsqrt(variance + self.eps)
        # on repasse dans le type d'origine avant d'appliquer le poids
        return self.weight * x_norm.to(orig_dtype)

class SwiGLU(nn.Module):
    def __init__(self, dim, inter_dim=None, dropout=0.1): # Ajout du paramètre dropout
        super().__init__()
        # Si aucun inter_dim n'est donné, on utilise la formule standard de LLaMA
        self.inter_dim = inter_dim if inter_dim is not None else int(dim * 4 * 2 / 3)
        # Une seule couche linéaire qui remplace w1 et w2 pour optimiser le GEMM (General Matrix to Matrix Multiplication)
        self.w12 = nn.Linear(dim, self.inter_dim * 2)
        self.w3 = nn.Linear(self.inter_dim, dim)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x):
        # on projette en une seule fois (1 seule multiplication matricielle au lieu de 2)
        x12 = self.w12(x)
        # on découpe le tenseur en deux portions égales sur la dernière dimension
        x1, x2 = x12.chunk(2, dim=-1)
        # on applique la formule SwiGLU + Dropout + Projection finale
        return self.w3(self.dropout(F.silu(x1) * x2))

In [ ]:
# =============================================================================
# 1. Radial Basis Function 
# =============================================================================
class RadialBasisFunction(nn.Module):
    def __init__(self, num_gbasis=16, r_max=8.0):
        super().__init__()
        self.num_gbasis = num_gbasis
        self.r_max = r_max

        # on initialise les centres et les largeurs, mais en nn.Parameter
        # Le modèle va ajuster sa rétine sur les échelles de distance clés du dataset
        initial_centers = torch.linspace(0.0, r_max, num_gbasis)
        initial_widths = torch.ones(num_gbasis) * (r_max / num_gbasis)
        
        self.centers = nn.Parameter(initial_centers)
        # epsilon de sécurité 1e-6 pour éviter toute division par zéro
        self.widths = nn.Parameter(initial_widths)

    def forward(self, dist):
        # 1. Calcul des Gaussiennes apprenables
        # on ajoute un epsilon de sécurité au dénominateur
        gaussians = torch.exp(-((dist - self.centers) ** 2) / (self.widths.pow(2) + 1e-6))
        
        # 2. Cosine Cutoff) 
        # f_c(d) = 0.5 * (cos(pi * d / r_max) + 1)
        # on clampe la distance au maximum à r_max pour éviter les retours de phase du cosinus
        clamped_dist = torch.clamp(dist, max=self.r_max)
        cutoff = 0.5 * (torch.cos(clamped_dist * torch.pi / self.r_max) + 1.0)
        
        # on multiplie l'activation par l'enveloppe de coupure
        # Garantit que les arêtes s'éteignent en douceur à la frontière des 8.0 Å [7]
        return gaussians * cutoff

# =============================================================================
# 2. EGNN LAYER (Avec Dampening 0.1)
# =============================================================================
class AEGIS_GT_Layer(nn.Module):
    def __init__(self, d_model, update_coords=False):
        super().__init__()
        self.update_coords = update_coords

        # 16 bases radiales de 0 à 8.0 Å
        self.rbf = RadialBasisFunction(num_gbasis=16, r_max=8.0)

        # Norms de sécurité (Pre-Norm) pour stabiliser le SwiGLU
        self.norm_edge_in = RMSNorm(2 * d_model + 16)
        self.norm_node_in = RMSNorm(2 * d_model)
        
        self.edge_mlp = nn.Sequential(nn.Linear(2 * d_model + 16, d_model), SwiGLU(d_model, inter_dim=d_model, dropout=0.1))
        self.node_mlp = nn.Sequential(nn.Linear(2 * d_model, d_model), SwiGLU(d_model, inter_dim=d_model, dropout=0.1))
        self.norm = RMSNorm(d_model)
        if update_coords:
            self.coord_mlp = nn.Sequential(nn.Linear(d_model, d_model // 4), nn.SiLU(), nn.Linear(d_model // 4, 1), nn.Tanh())

    def forward(self, h, x, edge_index):
        if edge_index.numel() == 0: 
            return h, x
        
        row, col = edge_index

        # on effectue une seule soustraction et indexation en mémoire
        coord_diff = x[row] - x[col]
        coord_diff_f32 = coord_diff.to(torch.float32) # Passage en float32 pour la stabilité de la norme

        dist_sq = torch.sum(coord_diff_f32.pow(2), dim=-1, keepdim=True).clamp(min=1e-6, max=1e3)
        #dist_embed = torch.log1p(dist_sq).to(x.dtype) # on revient en bfloat16 pour la suite
        dist = torch.sqrt(dist_sq) # [Edges, 1]
        dist_embed = self.rbf(dist).to(h.dtype) # Shape: [Edges, 16]

        # Message avec Pre-Norm
        edge_input = torch.cat([h[row], h[col], dist_embed], dim=-1)
        msg = self.edge_mlp(self.norm_edge_in(edge_input))
        
        if self.update_coords:
            # on réutilise directement 'coord_diff' calculé précédemment
            pos_update = scatter_mean(coord_diff * self.coord_mlp(msg), row, dim=0, dim_size=x.size(0))
            x = x + pos_update * 0.1 # Dampening de 0.1 pour la stabilité
        # Node update avec Pre-Norm
        node_input = torch.cat([h, scatter_mean(msg, row, dim=0, dim_size=h.size(0))], dim=-1)
        h = h + self.node_mlp(self.norm_node_in(node_input))
        
        return self.norm(h), x

# =============================================================================
# 3. Cross-Attention avec distance bias 
# =============================================================================
class AEGIS_GT_CrossAttention(nn.Module):
    def __init__(self, h_dim, dist_temp=12.0):
        super().__init__()
        self.h_dim = h_dim
        self.head_dim = 64
        self.num_heads = h_dim // self.head_dim 

        self.dist_temp = nn.Parameter(torch.tensor([float(dist_temp)])) 
        
        # on remplace nn.MHA par des projections manuelles pour utiliser SDPA (Single Dot Product Attention)
        self.q_proj = nn.Linear(h_dim, h_dim)
        self.kv_proj = nn.Linear(h_dim, h_dim * 2)
        self.out_proj = nn.Linear(h_dim, h_dim)
        
        self.norm = RMSNorm(h_dim)

    def forward(self, hp, hl, p_batch, l_batch, pos_p, pos_l):
        # 1. Conversion en format Dense
        hp_d, p_mask = to_dense_batch(hp, p_batch) # [B, Seq_P, D]
        hl_d, l_mask = to_dense_batch(hl, l_batch) # [B, Seq_L, D]
        pp_d, _ = to_dense_batch(pos_p, p_batch)   # [B, Seq_P, 3]
        pl_d, _ = to_dense_batch(pos_l, l_batch)   # [B, Seq_L, 3]

        # 2. Calcul de la distance en float32
        # on force le calcul en f32 pour éviter les NaNs en bfloat16
        dist = torch.cdist(pl_d.float(), pp_d.float()) 
        
        # Softplus pour garantir une température toujours positive > 0
        safe_temp = F.softplus(self.dist_temp) + 1e-6
        bias = (-dist / safe_temp).to(hl_d.dtype) # on revient au dtype du modèle
        
        # 3. Projections Q, K, V pour l'attention
        # Forme cible : [B, num_heads, Seq, head_dim]
        q = self.q_proj(hl_d).view(hl_d.size(0), hl_d.size(1), self.num_heads, self.head_dim).transpose(1, 2)

        # Projection KV unique pour la protéine
        kv = self.kv_proj(hp_d) # [B, Seq_P, D * 2]
        k_raw, v_raw = kv.chunk(2, dim=-1) # Découpage en K et V
        
        k = k_raw.view(hp_d.size(0), hp_d.size(1), self.num_heads, self.head_dim).transpose(1, 2)
        v = v_raw.view(hp_d.size(0), hp_d.size(1), self.num_heads, self.head_dim).transpose(1, 2)
        
        # 4. Préparation du bias et du masque
        # bias shape: [B, Seq_L, Seq_P] -> on ajoute la dim des têtes : [B, 1, Seq_L, Seq_P]
        # SDPA fera le broadcasting automatique sur les 'num_heads' sans copier les données
        bias = bias.unsqueeze(1) 
        
        # Masque de padding pour la protéine (key_padding_mask)
        # p_mask: [B, Seq_P] -> [B, 1, 1, Seq_P]
        p_attn_mask = p_mask.unsqueeze(1).unsqueeze(2)

        # on force les scores d'attention des nœuds de padding à une valeur très basse
        # -1e4 pour float16 (pour éviter l'overflow négatif) ou -1e9 pour bfloat16/float32
        pad_value = -1e4 if hl_d.dtype == torch.float16 else -1e9
        combined_mask = bias.masked_fill(~p_attn_mask, pad_value)

        # 5. Flash Attention / SDPA
        # on combine le bias de distance et le masque de padding
        # on utilise l'astuce : mask = (mask_bool) + bias
        attn_out = torch.nn.functional.scaled_dot_product_attention(
            q, k, v, 
            attn_mask=combined_mask, 
            dropout_p=0.0, 
            is_causal=False
        )
        
        # 6. Reprojection et Norm
        # [B, num_heads, Seq_L, head_dim] -> [B, Seq_L, D]
        attn_out = attn_out.transpose(1, 2).contiguous().view(hl_d.size(0), hl_d.size(1), -1)
        attn_out = self.out_proj(attn_out)
        
        return self.norm(attn_out[l_mask])           

In [ ]:
# =============================================================================
# 4. Pooling intelligent (Gated Attention)
# =============================================================================
class BillionScaleGatedPooling(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.d_model = d_model
        self.gate_dim = d_model // 2
        
        # Fusion des premières projections linéaires
        self.fused_proj = nn.Linear(d_model, self.gate_dim + d_model)
        self.gate_linear2 = nn.Linear(self.gate_dim, 1)
        self.norm = RMSNorm(d_model)
        
    def forward(self, x, batch):
        # Projection unique (1 seule opération GEMM sur le GPU)
        proj = self.fused_proj(x)
        
        # Découpage du tenseur fusionné
        gate_in, feat_in = proj.split([self.gate_dim, self.d_model], dim=-1)
        
        # Passage dans les fonctions d'activation respectives
        gate_out = self.gate_linear2(F.silu(gate_in))
        feat_out = F.silu(feat_in)
        
        # Calcul de l'attention Softmax et réduction
        weights = scatter_softmax(gate_out, batch, dim=0)
        pooled = scatter_add(weights * feat_out, batch, dim=0)
        
        return self.norm(pooled), weights

# =============================================================================
# 5. Architecture finale AEGIS_GT
# =============================================================================
class AEGIS_GT_Block_Logic(nn.Module):
    
    def __init__(self, d_model):
        super().__init__()
        self.d_model = d_model
        self.head_dim = 64
        self.num_heads = d_model // self.head_dim

        # Le biais spatial apprenable (1 scalaire par tête d'attention)
        self.spatial_bias = nn.Parameter(torch.zeros(1, self.num_heads, 1, 1))
        
        # Projections d'attention (plus rapide que nn.MHA)
        self.qkv_p = nn.Linear(d_model, d_model * 3)
        self.out_p = nn.Linear(d_model, d_model)
        self.egnn_p = AEGIS_GT_Layer(d_model, update_coords=False)

        # Modules Ligands (Instanciés mais utilisés uniquement si hl_d n'est pas None)
        self.qkv_l = nn.Linear(d_model, d_model * 3)
        self.out_l = nn.Linear(d_model, d_model)
        self.egnn_l = AEGIS_GT_Layer(d_model, update_coords=True)
        
        self.ff = SwiGLU(d_model, dropout=0.1)
        self.n = RMSNorm(d_model)
    
    def forward(self, hp_d, p_mask, ep, xp_d, hl_d=None, l_mask=None, el=None, xl_d=None, p_batch=None):
        
        # Flux Protéine  
        # 1. Création du masque de Padding en Float
        # 0.0 pour les vrais acides aminés, -infinity pour le padding
        pad_value = -1e4 if hp_d.dtype == torch.float16 else -1e9
        float_attn_mask = torch.zeros_like(p_mask, dtype=hp_d.dtype)
        float_attn_mask = float_attn_mask.masked_fill(~p_mask, pad_value)
        float_attn_mask = float_attn_mask.unsqueeze(1).unsqueeze(2) # [B, 1, 1, L]

        # 2. Injection du Spatial Bias via les arêtes (ep)
        if ep is not None and ep.numel() > 0 and p_batch is not None:
            # on convertit les arêtes sparse en matrice d'adjacence dense [B, L, L]
            adj = to_dense_adj(ep, batch=p_batch, max_num_nodes=hp_d.size(1))
            adj = adj.unsqueeze(1) # [B, 1, L, L]
            
            # on ajoute le biais appris partout où adj == 1
            float_attn_mask = float_attn_mask + (adj * self.spatial_bias)

        # 3. Calcul Q, K, V
        qkv_p = self.qkv_p(self.n(hp_d)).reshape(hp_d.size(0), hp_d.size(1), 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        # on force la contiguïté mémoire de Q, K, V
        q_p, k_p, v_p = qkv_p[0].contiguous(), qkv_p[1].contiguous(), qkv_p[2].contiguous()
        # Flash Attention avec le dropout intégré 
        hp_attn = torch.nn.functional.scaled_dot_product_attention(
            q_p, k_p, v_p, 
            attn_mask=float_attn_mask,
            dropout_p=0.1 if self.training else 0.0  
        )
        hp_d = hp_d + self.out_p(hp_attn.permute(0, 2, 1, 3).reshape(hp_d.size(0), hp_d.size(1), -1))

        # EGNN Protéine
        hp_s = hp_d[p_mask]
        xp_s = xp_d[p_mask]
        hp_n, _ = self.egnn_p(hp_s, xp_s, ep)
        
        hp_out = hp_d.clone()
        hp_out[p_mask] = hp_s + self.ff(hp_n)

        # Flux Ligand (Actif uniquement en phase 2)
        hl_out, xl_out = None, None

        if hl_d is not None and l_mask is not None:
            
            l_float_mask = torch.zeros_like(l_mask, dtype=hl_d.dtype)
            l_float_mask = l_float_mask.masked_fill(~l_mask, pad_value)
            l_float_mask = l_float_mask.unsqueeze(1).unsqueeze(2)
            
            # 1. Calcul Q, K, V
            qkv_l = self.qkv_l(self.n(hl_d)).reshape(hl_d.size(0), hl_d.size(1), 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
            # on force la contiguïté mémoire de Q, K, V
            q_l, k_l, v_l = qkv_l[0].contiguous(), qkv_l[1].contiguous(), qkv_l[2].contiguous()
            # Flash Attention avec le dropout intégré
            hl_attn = torch.nn.functional.scaled_dot_product_attention(
                q_l, k_l, v_l, 
                attn_mask=l_float_mask, 
                dropout_p=0.1 if self.training else 0.0 
            )
            
            hl_d = hl_d + self.out_l(hl_attn.permute(0, 2, 1, 3).reshape(hl_d.size(0), hl_d.size(1), -1))
        
            # EGNN Ligand 
            # on extrait les nodes pour le calcul géométrique
            hl_s = hl_d[l_mask]
            xl_s = xl_d[l_mask]
            hl_n, xl_new_s = self.egnn_l(hl_s, xl_s, el)
        
            # 3. Mise à jour et Feed-Forward (Format DENSE) 
            # on utilise .clone() pour être compatible avec Gradient Checkpointing
        
            hl_out = hl_d.clone()
            xl_out = xl_d.clone()
        
            # Ré-injection du signal affiné par SwiGLU (self.ff)
            hl_out[l_mask] = hl_s + self.ff(hl_n)
            xl_out[l_mask] = xl_new_s # Mise à jour des positions du ligand

        return hp_out, hl_out, xl_out

In [ ]:
import torch
import torch.nn as nn
from torch_geometric.utils import to_dense_batch
from torch.utils.checkpoint import checkpoint

class AEGIS_GT(nn.Module):
    def __init__(self, p_dim=2589, l_dim=23, h_dim=768, n_layers=6, rdkit_dim=15):
        super().__init__()
        
        # 1. Encodeurs Initiaux
        self.p_proj = nn.Linear(p_dim, h_dim)
        self.l_proj = nn.Linear(l_dim, h_dim)

        self.physics_adapter = nn.Sequential(
            nn.Linear(6, h_dim // 2),
            SwiGLU(h_dim // 2, dropout=0.1),
            nn.Linear(h_dim // 2, h_dim)
        )

        nn.init.constant_(self.physics_adapter[-1].weight, 0.0)
        nn.init.constant_(self.physics_adapter[-1].bias, 0.0)
        
        # 2. Le Backbone Géométrique
        self.layers = nn.ModuleList([AEGIS_GT_Block_Logic(h_dim) for _ in range(n_layers)])
        
        # 3. Cross-Attention & Pooling (Réservé Phase 2)
        self.cross_attn_l2p = AEGIS_GT_CrossAttention(h_dim) 
        self.cross_attn_p2l = AEGIS_GT_CrossAttention(h_dim) 
        self.l_pool = BillionScaleGatedPooling(h_dim) # Uniquement pour le Ligand

        # ==========================================
        # 4. Tetes de prédiction de la phase 1 (Physique de la Mutation)
        # ==========================================

        # Remet la variance à 1.0 après les 6 couches
        self.phase1_final_norm = RMSNorm(h_dim)

        self.phase1_heads = nn.ModuleDict({
            'structural_rmsd_A': nn.Sequential(
                nn.Linear(h_dim, h_dim // 4), SwiGLU(h_dim // 4, dropout=0.1), nn.Linear(h_dim // 4, 1)
            ),
            'true_delta_sasa': nn.Sequential(
                nn.Linear(h_dim, h_dim // 4), SwiGLU(h_dim // 4, dropout=0.1), nn.Linear(h_dim // 4, 1)
            ),
            'true_delta_packing': nn.Sequential(
                nn.Linear(h_dim, h_dim // 4), SwiGLU(h_dim // 4, dropout=0.1), nn.Linear(h_dim // 4, 1)
            ),
            'true_delta_solv_hydro': nn.Sequential(
                nn.Linear(h_dim, h_dim // 4), SwiGLU(h_dim // 4, dropout=0.1), nn.Linear(h_dim // 4, 1)
            ),
            'true_delta_electro': nn.Sequential(
                nn.Linear(h_dim, h_dim // 4), SwiGLU(h_dim // 4, dropout=0.1), nn.Linear(h_dim // 4, 1)
            ),
            'true_delta_clash': nn.Sequential(
                nn.Linear(h_dim, h_dim // 4), SwiGLU(h_dim // 4, dropout=0.1), nn.Linear(h_dim // 4, 1)
            ),
            'true_delta_ddg': nn.Sequential(
                nn.Linear(h_dim, h_dim // 4), SwiGLU(h_dim // 4, dropout=0.1), nn.Linear(h_dim // 4, 1)
            )
        })

        # ==========================================
        # 5. Tetes de prédiction de la phase 2 (Affinité + Résistance clinique)
        # ==========================================
        # Fusion : Protéine (768) + Ligand (768) + RDKit (15) = 1551
        fusion_input_dim = (h_dim * 2) + rdkit_dim
        # on utilise SwiGLU pour la fusion, 
        self.fusion = nn.Sequential(
            nn.Linear(fusion_input_dim, 1024), 
            SwiGLU(1024, dropout=0.2), 
            RMSNorm(1024)
        )
        
        # Régression de l'affinité (delta_pAff) -> 1 dimension 
        self.phase2_pAff_head = nn.Sequential(
            nn.Linear(1024, 512),
            SwiGLU(512, dropout=0.1),
            nn.Linear(512, 1)
        )
        

    def forward(self, data):
        # ---------------------------------------------------------
        # Detection de la phase (Data = Phase 1, HeteroData = Phase 2)
        # ---------------------------------------------------------
        is_phase2 = hasattr(data, 'ligand_global_feat')

        # 1. Extraction Protéine (Commune aux deux phases)
        p_x = data['protein'].x if is_phase2 else data.x
        xp  = data['protein'].pos if is_phase2 else data.pos
        p_batch = data['protein'].batch if is_phase2 else getattr(data, 'batch', torch.zeros(p_x.size(0), dtype=torch.long, device=p_x.device))
        ep  = data['protein', 'interacts', 'protein'].edge_index if is_phase2 else data.edge_index
        local_mut_idx = data.local_mut_idx.view(-1) # Shape [Batch]
        
        
        if is_phase2:
            physics_features = p_x[:, 23:29] # Extraction des 6 métriques physiques
            p_x_base = p_x.clone()
            p_x_base[:, 23:29] = 0.0         # Le backbone continue de voir des zéros
            
            hp = self.p_proj(p_x_base)
            hp = hp + self.physics_adapter(physics_features) # on additionne le savoir
        else:
            hp = self.p_proj(p_x) # Phase 1 classique
            
        hp_d, p_mask = to_dense_batch(hp, p_batch)
        xp_d, _ = to_dense_batch(xp, p_batch)
        
        # Coordinate jittering
        if self.training:
            # on ajoute un micro-bruit de 0.12 Å uniquement pendant l'entraînement pour empêcher la mémorisation de la géométrie 3D des protéines
            xp_d = xp_d + torch.randn_like(xp_d) * 0.12

        # Filet de sécurité arêtes Protéine
        if ep.numel() > 0:
            row_p, col_p = ep
            mask_p = (row_p >= 0) & (row_p < hp.size(0)) & (col_p >= 0) & (col_p < hp.size(0))
            ep = ep[:, mask_p]

        # 2. Extraction Ligand (Seulement si Phase 2)
        hl_d, xl_d, el, l_mask = None, None, None, None
        if is_phase2:
            hl = self.l_proj(data['ligand'].x)
            xl = data['ligand'].pos
            l_batch = data['ligand'].batch
            el = data['ligand', 'interacts', 'ligand'].edge_index
            
            if el.numel() > 0:
                row_l, col_l = el
                mask_l = (row_l >= 0) & (row_l < hl.size(0)) & (col_l >= 0) & (col_l < hl.size(0))
                el = el[:, mask_l]

            hl_d, l_mask = to_dense_batch(hl, l_batch)
            xl_d, _ = to_dense_batch(xl, l_batch)

        # ========================================================
        # 3. Le Backbone (Gradient Checkpointing Interleaved)
        # ========================================================
        for i, layer in enumerate(self.layers):
            if i % 2 == 1 and self.training:
                # on doit passer les arguments dans l'ordre exact de la classe BGTBlock_Logic
                hp_d, hl_d, xl_d = torch.utils.checkpoint.checkpoint(
                    layer, hp_d, p_mask, ep, xp_d, hl_d, l_mask, el, xl_d, p_batch,
                    use_reentrant=False
                )
            else:
                hp_d, hl_d, xl_d = layer(hp_d, p_mask, ep, xp_d, hl_d, l_mask, el, xl_d, p_batch)


        # ========================================================
        # 5. Bifurcation des prédictions (Phase 1 vs Phase 2)
        # ========================================================
        outputs = {}

        if not is_phase2:

            # ========================================================
            # 4. Target Pooling (Extraction du vecteur de la mutation)
            # ========================================================
            # hp_d shape: [Batch, MaxSeq, 768]
            # on extrait précisément le vecteur à l'index de la mutation pour chaque élément du batch
            batch_indices = torch.arange(hp_d.size(0), device=hp_d.device)
            p_mut_vec = hp_d[batch_indices, local_mut_idx, :] # Shape: [Batch, 768]
            
            # Normalisation finale obligatoire
            p_mut_vec = self.phase1_final_norm(p_mut_vec) # Normalisation de sortie

            # Inférence propre et vectorisée par dictionnaire sur les 7 têtes indépendantes
            outputs = {name: head(p_mut_vec) for name, head in self.phase1_heads.items()}

            outputs['pred_deltas_tensor'] = torch.cat([
                outputs['structural_rmsd_A'], outputs['true_delta_sasa'], 
                outputs['true_delta_packing'], outputs['true_delta_solv_hydro'], 
                outputs['true_delta_electro'], outputs['true_delta_clash'], 
                outputs['true_delta_ddg']
            ], dim=-1) # Shape final: [B, 7]
            
            return outputs, None, None

        else:
            # En mode phase 2 
            hp_sparse = hp_d[p_mask]
            hl_sparse = hl_d[l_mask]
            xl_sparse = xl_d[l_mask]
            xp_sparse = xp_d[p_mask]

            # Cross-Attention
            # le ligand regarde la protéine
            hl_final = self.cross_attn_l2p(hp_sparse, hl_sparse, data['protein'].batch, data['ligand'].batch, xp_sparse, xl_sparse)

            # La protéine regarde le ligand
            hp_final = self.cross_attn_p2l(hl_sparse, hp_sparse, data['ligand'].batch, data['protein'].batch, xl_sparse, xp_sparse)


            # 2. Target pooling (sur la protéine qui a vu le ligand)
            # on remet la protéine en Dense pour pouvoir utiliser l'index de la mutation
            hp_final_d, _ = to_dense_batch(hp_final, data['protein'].batch)
            batch_indices = torch.arange(hp_final_d.size(0), device=hp_final_d.device)
            p_mut_vec_aware = hp_final_d[batch_indices, local_mut_idx, :] # Le vecteur est "conscient" du ligand !  
            
            # Gated pooling (sur le ligand qui a vu la protéine)
            l_vec, l_w = self.l_pool(hl_final, data['ligand'].batch)
            
            # Fusion : Target Pooled Protein + Gated Pooled Ligand + RDKit
            rdkit_global = data.ligand_global_feat.view(p_mut_vec_aware.size(0), -1)
            combined = torch.cat([p_mut_vec_aware, l_vec, rdkit_global], dim=-1)
            
            feat = self.fusion(combined)
            
            outputs = {
                'delta_pAff': self.phase2_pAff_head(feat)
            }
            return outputs, None, l_w # p_w n'est plus pertinent via Target Pooling
            
    def prepare_for_phase2(self):
        """
        Gèle le savoir de la protéine mutée (Phase 1) mais garde 
        actifs le ligand, la physique et la fusion pour la Phase 2.
        """
        #1. on gèle tout par défaut (le backbone Phase 1)
        for param in self.parameters():
            param.requires_grad = False
            
        # 2. on débloque l'adaptateur physique 
        for param in self.physics_adapter.parameters():
            param.requires_grad = True
            
        # 3. on débloque l'encodeur du Ligand
        for param in self.l_proj.parameters():
            param.requires_grad = True
            
        # 4. on débloque uniquement la partie Ligand dans le Backbone
        for layer in self.layers:
            for param in layer.qkv_l.parameters(): param.requires_grad = True
            for param in layer.out_l.parameters(): param.requires_grad = True
            for param in layer.egnn_l.parameters(): param.requires_grad = True
            
        # 5. on débloque la Cross-Attention et le Pooling Phase 2
        for param in self.cross_attn_l2p.parameters(): param.requires_grad = True
        for param in self.cross_attn_p2l.parameters(): param.requires_grad = True
        for param in self.l_pool.parameters(): param.requires_grad = True
        
        # 6. on débloque les têtes de prédiction Phase 2
        for param in self.fusion.parameters(): param.requires_grad = True
        for param in self.phase2_pAff_head.parameters(): param.requires_grad = True
        
        # Affichage du statut dans la console
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        frozen = sum(p.numel() for p in self.parameters() if not p.requires_grad)
        print(f"Modèle préparé pour la Phase 2.")
        print(f"➜ Paramètres gelés (Savoir Phase 1) : {frozen:,}")
        print(f"➜ Paramètres entraînables (Ligand + Fusion) : {trainable:,}")
    
          

In [ ]:
# =============================================================================
# 6. Pairwise Ranking Loss
# =============================================================================
def pairwise_ranking_loss(pred, target, genes, margin=0.1):
    """
    Perte de classement par paires. 
    l'objectif est que si Target_A > Target_B, alors Pred_A > Pred_B.
    Sert à affiner la direction de l'affinité (Phase 2).
    """
    # Squeeze pour s'assurer d'avoir des vecteurs 1D
    pred = pred.view(-1)
    target = target.detach().view(-1)

    # s'assurer que genes est un tenseur 1D (ex: des IDs de gènes entiers)
    genes = genes.view(-1)
    
    if pred.size(0) < 2: 
        return torch.tensor(0.0, device=pred.device, requires_grad=True)
    
    # Calcul des différences par paires (Matrice N x N)
    # on utilise le broadcasting pour créer la matrice de différences
    diff_pred = pred.unsqueeze(1) - pred.unsqueeze(0)
    diff_target = target.unsqueeze(1) - target.unsqueeze(0)
    
    # Masque pour ne considérer que les paires où la différence de cible est significative (supérieure à la marge)
    mask_margin = diff_target.abs() > margin 

    # Masque de Gène (Séquence d'identité)
    # on crée une matrice booléenne où [i, j] est True si gene[i] == gene[j]
    gene_mask = (genes.unsqueeze(1) == genes.unsqueeze(0))

    # on combine les deux masques, même gène et différence significative
    final_mask = mask_margin & gene_mask
    
    # Perte de classement : on pénalise si la direction de la prédiction est opposée à la direction de la cible
    # Formule : relu(margin - sign(diff_target) * diff_pred)
    loss = F.relu(margin - diff_target.sign() * diff_pred)
    
    # Moyenne pondérée par le masque
    return (loss * final_mask.float()).sum() / (final_mask.float().sum() + 1e-6)

# =============================================================================
# 3. Initialisation des Poids (Kaiming/Xavier)
# =============================================================================
def init_weights(m):
    if isinstance(m, nn.Linear):
        # Têtes de sortie (Linear heads) 
        # Pour les sorties finales (Classification ou Régression ), 
        # on utilise Xavier pour une distribution centrée et stable.
        if m.out_features in [1, 3, 7, 8]:
            nn.init.xavier_uniform_(m.weight)
        
        # Couches Internes (Séquence/Graphe/Fusion) 
        # Pour tout le reste, on utilise Kaiming (He) car on utilise les fonctions d'activation SiLU/Swiglu.
        # Cela évite la disparition du gradient dans les réseaux profonds.
        else:
            nn.init.kaiming_uniform_(m.weight, nonlinearity='relu')
        
        # Initialisation du biais à zéro pour éviter tout décalage initial
        if m.bias is not None:
            nn.init.constant_(m.bias, 0)


In [ ]:
import torch
import logging

if __name__ == "__main__":
    
    # 1. Création du modèle en Float32 
    model = AEGIS_GT(p_dim=2589, h_dim=256, n_layers=4)
    
    # 2. Initialisation statistique des poids en Float32
    model.apply(init_weights)
    
    # 3. Transfert vers le GPU en float32 (l'autocast se chargera du bfloat16 plus tard) 
    model = model.to("cuda")
    
    # 4. Mesure et affichage des paramètres
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Le modèle AEGIS_GT est initialisé !")
    print(f"Le nombre de paramètres entraînables est : {total_params:,}")

In [ ]:
import graphviz
from IPython.display import display

def draw_bgt_architecture():
    # Création du graphe dirigé
    dot = graphviz.Digraph('AEGIS_GT', comment='Affinity Evaluation & Geometric Induced-fit Screening', format='png')
    dot.attr(rankdir='TB', size='12,15', fontname='Helvetica', fontsize='14', nodesep='0.6', ranksep='0.8')
    
    # Styles globaux
    dot.attr('node', shape='box', style='filled, rounded', fontname='Helvetica', fontsize='12', margin='0.2')
    dot.attr('edge', fontname='Helvetica', fontsize='10', color='#555555')

    # ==========================================
    # 1. ENTRÉES (INPUTS)
    # ==========================================
    with dot.subgraph(name='cluster_inputs') as c:
        c.attr(label="Données d'Entrée (Inputs)", style='dashed', color='grey', bgcolor='#f9f9f9')
        # Protéine divisée proprement pour l'adaptation physique
        c.node('P_Seq_Feat', 'Séquence Protéine\n[2583 dims]\n(ESM2 + pLDDT + OneHot)', fillcolor='#d0e8f2', shape='cylinder')
        c.node('P_Phys_Feat', 'Métriques Physiques\n[6 dims]\n(SASA, Packing, ddG...)', fillcolor='#bbdefb', shape='cylinder')
        c.node('P_Geom', 'Géométrie Protéine\n[Coords 3D + Edges]', fillcolor='#d0e8f2', shape='cylinder')
        
        # Ligand
        c.node('L_Feat', 'Features Ligand\n[23 dims]\n(Gasteiger, Types...)', fillcolor='#ffe4c4', shape='cylinder')
        c.node('L_Geom', 'Géométrie Ligand\n[Coords 3D + Edges]', fillcolor='#ffe4c4', shape='cylinder')
        c.node('L_Global', 'Features Globales RDKit\n[15 dims]', fillcolor='#ffe4c4', shape='cylinder')

    # ==========================================
    # 2. ENCODEURS (PROJECTIONS / ADAPTATION)
    # ==========================================
    dot.node('P_Proj', 'Projection Linéaire Protéine\n(Linear: 2589 → h_dim)', fillcolor='#a6dcef')
    dot.node('Phys_Adapt', 'Adaptateur Physique (Phase 2)\n(Linear + SwiGLU: 6 → h_dim)', fillcolor='#90caf9')
    dot.node('P_Sum', 'Sommation des Embeddings\nhp = hp_base + hp_phys', fillcolor='#e3f2fd', shape='circle')
    
    dot.node('L_Proj', 'Projection Linéaire Ligand\n(Linear: 23 → h_dim)', fillcolor='#ffcba4')

    # Connexions des Projections
    dot.edge('P_Seq_Feat', 'P_Proj', label=' hp_base')
    dot.edge('P_Phys_Feat', 'Phys_Adapt', label=' Phase 2 Only')
    dot.edge('P_Proj', 'P_Sum')
    dot.edge('Phys_Adapt', 'P_Sum')
    
    dot.edge('L_Feat', 'L_Proj', label=' Phase 2 Only')

    # ==========================================
    # 3. BACKBONE (4 LAYERS)
    # ==========================================
    with dot.subgraph(name='cluster_backbone') as c:
        c.attr(label='Backbone Géométrique (x4 Layers)\nGradient Checkpointing', style='solid', color='purple', bgcolor='#f3e8ff', penwidth='2')
        
        c.node('SpatialBias', 'Spatial Bias (Arêtes)\n[to_dense_adj]', fillcolor='#e9d5ff')
        c.node('P_Attn', 'Flash Attention (SDPA)\nProtéine (Self)', fillcolor='#d8b4fe')
        c.node('P_EGNN', 'EGNN Layer (Protéine)\n[Sans update_coords]', fillcolor='#c084fc')
        c.node('P_SwiGLU', 'RMSNorm + SwiGLU\n[Dropout 0.1]', fillcolor='#a855f7', fontcolor='white')

        c.node('L_Attn', 'Flash Attention (SDPA)\nLigand (Self)', fillcolor='#d8b4fe')
        c.node('L_EGNN', 'EGNN Layer (Ligand)\n[Avec update_coords]', fillcolor='#c084fc')
        c.node('L_SwiGLU', 'RMSNorm + SwiGLU\n[Dropout 0.1]', fillcolor='#a855f7', fontcolor='white')

    # Flux Protéine Backbone
    dot.edge('P_Sum', 'SpatialBias')
    dot.edge('P_Geom', 'SpatialBias', label=' ep')
    dot.edge('SpatialBias', 'P_Attn', label=' mask + bias')
    dot.edge('P_Attn', 'P_EGNN')
    dot.edge('P_Geom', 'P_EGNN', label=' coords')
    dot.edge('P_EGNN', 'P_SwiGLU')

    # Flux Ligand Backbone
    dot.edge('L_Proj', 'L_Attn')
    dot.edge('L_Attn', 'L_EGNN')
    dot.edge('L_Geom', 'L_EGNN', label=' coords')
    dot.edge('L_EGNN', 'L_SwiGLU')

    # ==========================================
    # 4. EXTRACTION CHIRURGICALE (PHASE 1)
    # ==========================================
    dot.node('TargetPool_1', 'Target Pooling (Phase 1)\nExtraction Index Mutation\n[Batch, h_dim]', fillcolor='#ffb347', shape='diamond')
    dot.edge('P_SwiGLU', 'TargetPool_1', label=' Phase 1 (No Ligand)')

    # ==========================================
    # 5. PHASE 1 HEADS
    # ==========================================
    with dot.subgraph(name='cluster_phase1') as c:
        c.attr(label='Phase 1 : Lois de la Physique', style='dashed', color='red', bgcolor='#ffeded')
        c.node('P1_Norm', 'RMSNorm', fillcolor='#ff9999')
        c.node('P1_MLP', 'SwiGLU MLP\n[h_dim → h_dim//4 → 7]', fillcolor='#ff6666', fontcolor='white')
        c.node('P1_Out', '7 Régressions Cibles\n(RMSD, SASA, Packing, DDG...)', fillcolor='#ff3333', fontcolor='white', shape='note')

    dot.edge('TargetPool_1', 'P1_Norm')
    dot.edge('P1_Norm', 'P1_MLP')
    dot.edge('P1_MLP', 'P1_Out')

    # ==========================================
    # 6. PHASE 2 : CROSS-ATTENTION (Symétrique)
    # ==========================================
    with dot.subgraph(name='cluster_cross') as c:
        c.attr(label='Phase 2 : Ajustement Induit (Induced Fit)', style='dashed', color='green', bgcolor='#e8f5e9')
        c.node('Cross_L2P', 'Cross-Attention L2P\n(Ligand regarde Protéine)', fillcolor='#a5d6a7')
        c.node('Cross_P2L', 'Cross-Attention P2L\n(Protéine regarde Ligand)', fillcolor='#a5d6a7')
        
    dot.edge('P_SwiGLU', 'Cross_L2P', label=' K, V')
    dot.edge('L_SwiGLU', 'Cross_L2P', label=' Q')
    
    dot.edge('L_SwiGLU', 'Cross_P2L', label=' K, V')
    dot.edge('P_SwiGLU', 'Cross_P2L', label=' Q')

    # ==========================================
    # 7. PHASE 2 : POOLING & FUSION
    # ==========================================
    dot.node('TargetPool_2', 'Target Pooling (Phase 2)\nProtéine Consciente du Ligand\n[Batch, h_dim]', fillcolor='#ffb347', shape='diamond')
    dot.node('GatedPool', 'Gated Pooling\nRéduction Séquence Ligand\n[Batch, h_dim]', fillcolor='#ffb347', shape='diamond')
    
    dot.edge('Cross_P2L', 'TargetPool_2')
    dot.edge('Cross_L2P', 'GatedPool')
    dot.node('Concat', 'Concaténation (Fusion)\n[h_dim*2 + 15 = 527 dims]', fillcolor='#cccccc', shape='invhouse')
    dot.edge('TargetPool_2', 'Concat')
    dot.edge('GatedPool', 'Concat')
    dot.edge('L_Global', 'Concat', style='dashed')

    # ==========================================
    # 8. PHASE 2 HEADS 
    # ==========================================
    with dot.subgraph(name='cluster_phase2') as c:
        c.attr(label='Phase 2 : Régression d\'Affinité Pure', style='dashed', color='blue', bgcolor='#e3f2fd')
        c.node('P2_Fusion_MLP', 'Fusion SwiGLU + RMSNorm\n[527 → 1024]', fillcolor='#90caf9')
        c.node('P2_pAff_MLP', 'pAff SwiGLU Head\n[1024 → 512]', fillcolor='#64b5f6')
        c.node('P2_Out1', 'delta_pAff (Régression)\n[1 dim]', fillcolor='#1e88e5', fontcolor='white', shape='note')

    dot.edge('Concat', 'P2_Fusion_MLP')
    dot.edge('P2_Fusion_MLP', 'P2_pAff_MLP')
    dot.edge('P2_pAff_MLP', 'P2_Out1')

    # Rendu et affichage
    dot.render('AEGIS_GT_Architecture', view=False)
    display(dot)

# Exécution de la fonction pour dessiner le graphe
draw_bgt_architecture()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import colorsys

# 1. Configuration de l'environnement graphique Premium (Échelle augmentée à 24 pour la régression pure)
fig, ax = plt.subplots(figsize=(24, 12), dpi=300) # Haute résolution
ax.set_xlim(-1, 24)
ax.set_ylim(-1, 11)
ax.axis('off')

# Convertisseur Hex -> RGB
def hex_to_rgb(hex_str):
    hex_str = hex_str.lstrip('#')
    return [int(hex_str[i:i+2], 16)/255.0 for i in (0, 2, 4)]

# Ajustement de la luminosité pour l'effet de matière 3D
def adjust_lightness(color, amount=0.5):
    rgb = hex_to_rgb(color)
    c = colorsys.rgb_to_hls(*rgb)
    new_rgb = colorsys.hls_to_rgb(c[0], max(0, min(1, c[1] * amount)), c[2])
    return new_rgb

# Fonction avancée pour dessiner un tenseur 3D ombré et élégant
def draw_premium_tensor(ax, x, y, w, h, d, color, label=None, dims=None):
    theta = np.radians(25)  # Angle isométrique doux
    dx = d * np.cos(theta) * 0.4
    dy = d * np.sin(theta) * 0.4
    
    # Déclinaisons de couleurs pour l'effet 3D
    c_top = adjust_lightness(color, 1.15)
    c_right = adjust_lightness(color, 0.85)
    c_border = adjust_lightness(color, 0.6)  # Bordure très fine de la même teinte
    
    # 1. Ombre portée au sol (Drop Shadow)
    sh_y = -0.6
    shadow = patches.Polygon([
        [x, y+sh_y], 
        [x+w, y+sh_y], 
        [x+w+dx, y+dy+sh_y], 
        [x+dx, y+dy+sh_y]
    ], facecolor='#e2e8f0', edgecolor='none', alpha=0.6, zorder=0)
    ax.add_patch(shadow)
    
    # 2. Face avant (Front)
    front = patches.Polygon([[x, y], [x+w, y], [x+w, y+h], [x, y+h]], 
                            facecolor=color, edgecolor=c_border, linewidth=0.5, zorder=2)
    ax.add_patch(front)
    
    # 3. Face supérieure (Top)
    top = patches.Polygon([[x, y+h], [x+dx, y+h+dy], [x+w+dx, y+h+dy], [x+w, y+h]], 
                                            facecolor=c_top, edgecolor=c_border, linewidth=0.5, zorder=2)
    ax.add_patch(top)
    
    # 4. Face latérale droite (Right)
    right = patches.Polygon([[x+w, y], [x+w+dx, y+dy], [x+w+dx, y+h+dy], [x+w, y+h]], 
                            facecolor=c_right, edgecolor=c_border, linewidth=0.5, zorder=2)
    ax.add_patch(right)
    
    # 5. Textes & Étiquettes
    if label:
        ax.text(x + w/2, y + h/2, label, ha='center', va='center', 
                fontsize=9, fontweight='bold', color='#1e293b', wrap=True, zorder=3)
        
    if dims:
        ax.text(x + w/2, y - 0.25, dims[0], ha='center', va='top', fontsize=7.5, color='#64748b', zorder=3)
        ax.text(x - 0.2, y + h/2, dims[1], ha='right', va='center', fontsize=7.5, color='#64748b', zorder=3)
        ax.text(x + w + dx/2 + 0.1, y + dy/2 - 0.1, dims[2], ha='left', va='center', 
                fontsize=7.5, color='#64748b', rotation=20, zorder=3)

# Fonction de connexion stylisée
def draw_premium_connection(ax, start, end, label=None, label_pos=None, style='-'):
    ax.annotate('', xy=end, xytext=start,
                arrowprops=dict(arrowstyle="->", lw=1.2, color='#64748b', ls=style,
                                shrinkA=4, shrinkB=4, mutation_scale=12))
    if label and label_pos:
        ax.text(label_pos[0], label_pos[1], label, fontsize=8, ha='center', va='bottom', color='#475569')

# =============================================================================
# Rendu de l'architecture AEGIS-GT (Régression Pure)
# =============================================================================

# Palette de couleurs "Modern Biotech"
C_PROT   = '#a5f3fc'  # Cyan protéine
C_PHYS   = '#93c5fd'  # Bleu clair adaptateur
C_LIG    = '#fed7aa'  # Orange pastel ligand
C_RDKIT  = '#fbcfe8'  # Rose RDKit
C_BACK   = '#ddd6fe'  # Violet clair backbone
C_CROSS  = '#bbf7d0'  # Vert cross-attn
C_POOL   = '#fef08a'  # Jaune pooling
C_FUSION = '#cbd5e1'  # Gris fusion
C_P1_OUT = '#fca5a5'  # Rouge Phase 1
C_P2_OUT = '#93c5fd'  # Bleu Phase 2

# ==========================================
# 1. ENTRÉES (INPUTS)
# ==========================================
# Séquence de base Protéine
draw_premium_tensor(ax, x=0.5, y=8.4, w=1.5, h=1.2, d=0.4, color=C_PROT, 
                    label='Protein\nSequence\nFeatures', dims=['Batch', 'N_prot', '2583'])
# Métriques Physiques (FoldX/SASA)
draw_premium_tensor(ax, x=0.5, y=6.6, w=1.5, h=0.8, d=0.4, color=C_PHYS, 
                    label='Biophysical\nMetrics', dims=['Batch', 'N_prot', '6'])
# Coordonnées 3D Protéine
draw_premium_tensor(ax, x=0.5, y=4.8, w=1.5, h=0.8, d=0.4, color=C_PROT, 
                    label='Protein\nGeometry', dims=['Batch', 'N_prot', 'Coords'])

# Ligand Features
draw_premium_tensor(ax, x=0.5, y=3.0, w=1.5, h=1.0, d=0.4, color=C_LIG, 
                    label='Ligand\nFeatures', dims=['Batch', 'N_lig', '23'])
# Ligand Coordonnées 3D
draw_premium_tensor(ax, x=0.5, y=1.5, w=1.5, h=0.8, d=0.4, color=C_LIG, 
                    label='Ligand\nGeometry', dims=['Batch', 'N_lig', 'Coords'])
# RDKit Global
draw_premium_tensor(ax, x=0.5, y=0.1, w=1.5, h=0.5, d=0.4, color=C_RDKIT, 
                    label='RDKit\nGlobal', dims=['Batch', '1', '15'])

# ==========================================
# 2. ENCODEURS & PROJECTIONS
# ==========================================
# Projection Séquence
draw_premium_tensor(ax, x=4.0, y=8.4, w=1.5, h=0.8, d=0.4, color=C_PROT, 
                    label='Linear Proj.\n(2583 → 256)')
# Adaptateur Physique (Zéro-Init)
draw_premium_tensor(ax, x=4.0, y=6.6, w=1.5, h=0.8, d=0.4, color=C_PHYS, 
                    label='Physics Adapter\n(6 → 256)')
# Sommation des Embeddings
draw_premium_tensor(ax, x=6.2, y=7.7, w=1.0, h=0.8, d=0.4, color=C_FUSION, 
                    label='hp Sum')

# Projection Ligand
draw_premium_tensor(ax, x=4.0, y=3.0, w=1.5, h=0.8, d=0.4, color=C_LIG, 
                    label='Linear Proj.\n(23 → 256)')

# Connexions projections/adaptateur
draw_premium_connection(ax, (2.05, 9.0), (3.9, 9.0))
draw_premium_connection(ax, (2.05, 7.0), (3.9, 7.0), label='Phase 2 Only', label_pos=(3.0, 7.1))
draw_premium_connection(ax, (5.55, 8.8), (6.1, 8.35))
draw_premium_connection(ax, (5.55, 7.0), (6.1, 7.7))

draw_premium_connection(ax, (2.05, 3.5), (3.9, 3.5))

# ==========================================
# 3. BACKBONES (X4 LAYERS)
# ==========================================
# Protein Backbone (x4 Blocks)
draw_premium_tensor(ax, x=8.2, y=6.0, w=1.8, h=2.0, d=0.5, color=C_BACK, 
                    label='AEGIS\nProtein\nBackbone\n(4 Blocks)', dims=['Batch', 'N_prot', '256'])
# Ligand Backbone
draw_premium_tensor(ax, x=8.2, y=2.5, w=1.8, h=1.4, d=0.5, color=C_BACK, 
                    label='AEGIS\nLigand\nBackbone', dims=['Batch', 'N_lig', '256'])

draw_premium_connection(ax, (7.25, 8.1), (8.1, 7.6))
draw_premium_connection(ax, (2.05, 5.2), (8.1, 6.5), style='--') # Géométrie protéine
draw_premium_connection(ax, (5.55, 3.4), (8.1, 3.4))
draw_premium_connection(ax, (2.05, 1.9), (8.1, 2.9), style='--') # Géométrie ligand

# ==========================================
# 4. TARGET POOLING & CROSS-ATTENTION
# ==========================================
# Target Pooling Phase 1 (Foyer Mutation)
draw_premium_tensor(ax, x=12.2, y=7.2, w=1.2, h=1.2, d=0.4, color=C_POOL, 
                    label='Target\nPooling\n(Mutation)', dims=['Batch', '1', '256'])
# Symmetric Cross-Attention (Induced-Fit)
draw_premium_tensor(ax, x=12.2, y=2.5, w=1.8, h=1.5, d=0.5, color=C_CROSS, 
                    label='Symmetric\nCross-Attention\n(Induced-Fit)', dims=['Batch', 'N_lig', '256'])

draw_premium_connection(ax, (10.05, 7.0), (12.1, 7.8), label='Index Slicing', label_pos=(11.0, 7.9))
draw_premium_connection(ax, (10.05, 3.2), (12.1, 3.25)) # Ligand vers Cross-Attn
draw_premium_connection(ax, (9.1, 6.0), (12.1, 3.5), label='K, V from Protein', label_pos=(10.8, 4.85)) # Protéine vers Cross-Attn

# ==========================================
# 5. SORTIES PHASE 1 (Physique)
# ==========================================
draw_premium_tensor(ax, x=15.0, y=7.2, w=1.5, h=1.2, d=0.4, color=C_P1_OUT, 
                    label='Phase 1\nOutputs\n(7 Regressions)', dims=['Batch', '1', '7'])
draw_premium_connection(ax, (13.45, 7.8), (14.9, 7.8), label='Phase 1 Only', label_pos=(14.2, 7.9))

# ==========================================
# 6. POOLING PHASE 2 & CONCATÉNATION
# ==========================================
# Target Pooling Protéine (Consciente du Ligand)
draw_premium_tensor(ax, x=15.2, y=4.4, w=1.2, h=1.2, d=0.4, color=C_POOL, 
                    label='Target\nPooling\n(Aware)', dims=['Batch', '1', '256'])
# Gated Pooling Ligand
draw_premium_tensor(ax, x=15.2, y=2.5, w=1.2, h=1.2, d=0.4, color=C_POOL, 
                    label='Gated\nPooling', dims=['Batch', '1', '256'])

draw_premium_connection(ax, (14.05, 3.5), (15.1, 4.8))
draw_premium_connection(ax, (14.05, 3.1), (15.1, 3.1))

# Bloc de Concaténation (Fusion globale)
draw_premium_tensor(ax, x=18.0, y=2.4, w=1.4, h=1.4, d=0.4, color=C_FUSION, 
                    label='FUSION\n(Concat)', dims=['Batch', '1', '527'])

draw_premium_connection(ax, (16.45, 5.0), (17.9, 3.4), label='Mut_Vec', label_pos=(17.0, 4.5))
draw_premium_connection(ax, (16.45, 3.1), (17.9, 3.1), label='Lig_Vec', label_pos=(17.15, 3.2))

# RDKit Global vers la fusion (Ligne pointillée rose)
ax.annotate('', xy=(18.1, 2.3), xytext=(2.05, 0.35),
            arrowprops=dict(arrowstyle="->", lw=1.0, ls="--", color='#db2777', shrinkA=4, shrinkB=4))

# ==========================================
# 7. SORTIE PHASE 2 (Affinité delta_pAff)
# ==========================================
draw_premium_tensor(ax, x=21.0, y=2.8, w=1.4, h=1.0, d=0.3, color=C_P2_OUT, 
                    label='delta_pAff\nRegression\n(Pure)', dims=['Batch', '1', '1'])
draw_premium_connection(ax, (19.45, 3.1), (20.9, 3.1))

# Titre du diagramme
ax.text(11.5, 10.4, "AEGIS-GT: 3D Volumetric Tensor Flow Diagram (Pure Regression)", ha='center', va='center', 
        fontsize=18, fontweight='bold', color='#0f172a')

plt.tight_layout()
plt.savefig("aegis_volumetric_architecture_premium.png", dpi=300, bbox_inches='tight')
plt.show()


## **Partie Entrainement Phase 2 du modèle Transformer Géométrique**

In [ ]:
import torch
import os

# Chemin d'accès vers votre checkpoint de Phase 1 personnalisé
checkpoint_path = "/kaggle/input/bgt-checkpoint/aegis_best_ep41_step2700_mae1.0748.pth"

# Sécurités de chemin d'accès selon votre arborescence Kaggle
if not os.path.exists(checkpoint_path):
    checkpoint_path = "/kaggle/input/datasets/anisis/bgt-checkpoint/aegis_best_ep41_step2700_mae1.0748.pth"
if not os.path.exists(checkpoint_path):
    checkpoint_path = "aegis_best_ep41_step2700_mae1.0748.pth"

if os.path.exists(checkpoint_path):
    print(f"Analyse du checkpoint : {checkpoint_path}")
    
    # 1. Chargement sur le CPU pour économiser la mémoire du GPU
    checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    
    # 2. Inspection des métadonnées d'origine
    print("1. Métadonnées de l'Entraînement de Phase 1 :")
    print(f"Clés enregistrées dans le fichier : {list(checkpoint.keys())}")
    print(f"Époque de sauvegarde réelle  : {checkpoint.get('epoch', 'N/A')}")
    print(f"Étape globale de convergence : {checkpoint.get('step', 'N/A')}")
    print(f"Meilleure Perte enregistrée  : {checkpoint.get('best_score', 'N/A'):.4f}")
    
    
    # 3. Inspection de la structure des Tenseurs
    state_dict = checkpoint.get('model_state', {})
    print(f"2. Structure interne des paramètres (model_state) :")
    print(f"Nombre total de couches : {len(state_dict)}")
    
    # Séparation logique du Backbone et des têtes
    backbone_keys = [k for k in state_dict.keys() if k.startswith('p_proj') or k.startswith('layers')]
    phase1_head_keys = [k for k in state_dict.keys() if k.startswith('phase1_') or k.startswith('phase1_heads')]
    
    print(f"Nombre de tenseurs du Backbone : {len(backbone_keys)}")
    print(f"Nombre de tenseurs des têtes physiques : {len(phase1_head_keys)}")
    
    
    # 4. Validation des dimensions géométriques critiques
    print("3. Vérification des dimensions géométriques d'AEGIS-GT :")
    
    if 'p_proj.weight' in state_dict:
        w_shape = list(state_dict['p_proj.weight'].shape)
        print(f"p_proj.weight (Couche de projection) : {w_shape}")
        
    if 'layers.0.qkv_p.weight' in state_dict:
        w_shape = list(state_dict['layers.0.qkv_p.weight'].shape)
        print(f"layers.0.qkv_p.weight (Attention){w_shape}")
        
    if 'layers.0.spatial_bias' in state_dict:
        w_shape = list(state_dict['layers.0.spatial_bias'].shape)
        print(f"layers.0.spatial_bias (Biais spatial) : {w_shape}")
        
    if 'layers.0.ff.w12.weight' in state_dict:
        w_shape = list(state_dict['layers.0.ff.w12.weight'].shape)
        print(f"layers.0.ff.w12.weight : {w_shape}")

    print("Toutes les clés débutant par 'p_proj' et 'layers'. Leurs dimensions correspondent au modèle d'élite à 20M de paramètres.")
    print("Les clés 'phase1_final_norm' et 'phase1_heads'. Elles appartiennent à la Phase 1 et doivent être écartées.")
    print("Les nouvelles couches 'l_proj', 'cross_attn_l2p/p2l', 'l_pool' et 'fusion' doivent etre initialisées.")
    
else:
    print(f"Impossible de trouver le fichier de checkpoint à l'adresse : {checkpoint_path}")

In [ ]:
import torch
import os

print("Initialisation du modèle AEGIS pour le criblage clinique")

# 1. Instanciation du modèle avec les hyperparamètres optimisés (25M paramètres)
model_phase2 = AEGIS_GT(p_dim=2589, h_dim=256, n_layers=4)

# 2. Initialisation statistique globale (Xavier / Kaiming)
model_phase2.apply(init_weights)

# 3. Chemin du meilleur checkpoint de Phase 1
checkpoint_path = "/kaggle/input/datasets/anisis/bgt-checkpoint/aegis_best_ep41_step2700_mae1.0748.pth"

if os.path.exists(checkpoint_path):
    print(f"\nChargement des poids physiques depuis : {checkpoint_path}")
    checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    state_dict_p1 = checkpoint['model_state']
    
    # on filtre pour ne garder que le backbone partagé (p_proj et layers)
    # on exclut volontairement les têtes de Phase 1 (phase1_final_norm, phase1_deltas_head)
    backbone_state_dict = {
        k: v for k, v in state_dict_p1.items() 
        if k.startswith('p_proj') or k.startswith('layers')
    }
    
    # Chargement partiel (strict=False est obligatoire ici pour ignorer les couches manquantes du ligand)
    missing_keys, unexpected_keys = model_phase2.load_state_dict(backbone_state_dict, strict=False)
    print(f"le nombre de tenseurs physiques transférés : {len(backbone_state_dict)}")
    # on vérifie que les clés manquantes correspondent bien uniquement aux nouveaux modules du ligand
    print(f"les nouvelles couches initialisées en phase 2 sont : {[k for k in missing_keys if 'layers' not in k][:6]}")
else:
    print(f"Impossible de trouver le checkpoint de Phase 1 à l'adresse {checkpoint_path}.")

# Transfert final du modèle vers le GPU
model_phase2 = model_phase2.to("cuda")
model_phase2.prepare_for_phase2()

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# =============================================================================
# 1. Manager de perte Multi-Taches & Incertitude (Phase 2)
# =============================================================================
class Phase2LossManager(nn.Module):
    def __init__(self, device):
        super().__init__()
        # on utilise la Huber Loss pour la régression de l'affinité (robuste aux outliers)
        self.huber_loss_fn = nn.HuberLoss(reduction='none', delta=1.0)
        self.device = device
        
        # 3 paramètres apprenants pour équilibrer dynamiquement la Phase 2 :
        # Index 0 : delta_pAff (Régression d'affinité)
        # Index 2 : Pairwise Ranking (Classement par gène)
        self.log_vars = nn.Parameter(torch.zeros(2, device=device))

    def forward(self, preds_pAff, targets_pAff, genes, label_weight=None):
        # preds_pAff: [B, 1], targets_pAff: [B, 1]
        # genes: [B, 1]
        
        # 1. Calcul des pertes 
        # A. Régression d'affinité
        raw_pAff_loss = self.huber_loss_fn(preds_pAff.view(-1), targets_pAff.view(-1))
        
        # B. Perte de classement par paires 
        raw_rank_loss = pairwise_ranking_loss(preds_pAff, targets_pAff, genes)
        
        # 2. Application du poids de confiance clinique (label_weight de la Cellule 8)
        # Si un label clinique/structurel est peu fiable, son impact sur la loss est réduit !
        if label_weight is not None:
            label_weight = label_weight.to(preds_pAff.device).view(-1)
            raw_pAff_loss = raw_pAff_loss * label_weight
            
        # 3. Calcul des moyennes
        mean_pAff = raw_pAff_loss.mean()
        
        #total_loss = mean_pAff + 20.0 * raw_rank_loss
        total_loss = mean_pAff + raw_rank_loss
        
        losses = {
            'pAff_huber': mean_pAff.item(),
            'ranking_loss': raw_rank_loss.item()
        }
        
        return total_loss, losses

In [ ]:
import numpy as np
import torch
from scipy.stats import spearmanr, pearsonr  
from sklearn.metrics import mean_absolute_error
from tqdm.auto import tqdm
        
# =============================================================================
# 2. Moteur de validation
# =============================================================================
def validate_phase2(model, val_loader, loss_manager, device, max_batches=None):
    model.eval()
    
    all_true_pAff, all_pred_pAff = [], []
    
    total_val_loss = 0.0
    valid_batches = 0
    
    total_steps = len(val_loader)
    if max_batches is not None and max_batches < total_steps:
        total_steps = max_batches
        
    with torch.no_grad():
        for i, batch in enumerate(tqdm(val_loader, total=total_steps, desc="Évaluation Phase 2", leave=False)):
            if max_batches is not None and i >= max_batches:
                break
                
            batch = batch.to(device)
            with torch.amp.autocast('cuda', dtype=torch.bfloat16):
                outputs, _, _ = model(batch)
                
                preds_pAff = outputs['delta_pAff'] # [B, 1]
                targets_pAff = batch.y_delta_pAff.view(-1, 1) # [B, 1]
                
                # Calcul de la perte multi-tâches (Incertitude de Kendall + Ranking) 
                loss, _ = loss_manager(
                    preds_pAff, targets_pAff, 
                    batch.gene_id, batch.weight
                )
                
                total_val_loss += loss.item()
                valid_batches += 1
                
            # Collecte des prédictions de régression continue (pAff)
            all_true_pAff.append(targets_pAff.cpu().numpy())
            all_pred_pAff.append(preds_pAff.float().cpu().numpy())
                
    # Concaténation finale pour l'affinité
    y_true_pAff = np.concatenate(all_true_pAff).flatten()
    y_pred_pAff = np.concatenate(all_pred_pAff).flatten()
    
    pearson_corr, _  = pearsonr(y_true_pAff, y_pred_pAff)
    mae_pAff = mean_absolute_error(y_true_pAff, y_pred_pAff)
    
    # Calcul de la corrélation de Spearman 
    spearman_corr, _ = spearmanr(y_true_pAff, y_pred_pAff)
    
    metrics = {
        'Val_Loss': total_val_loss / max(1, valid_batches),
        'Pearson_pAff': pearson_corr,
        'Spearman_pAff': spearman_corr,  
        'MAE_pAff': mae_pAff
    }
        
    return metrics

# =============================================================================
# 3. Fonction d'affichage des métriques
# =============================================================================
def print_detailed_metrics_phase2(metrics):
    train_loss_str = f"Train Loss: {metrics.get('Train_Loss', 0.0):.4f} | " if 'Train_Loss' in metrics else ""
    val_loss_str = f"Val Loss: {metrics.get('Val_Loss', 0.0):.4f} | " if 'Val_Loss' in metrics else ""
    
    # 1. Affinité & Résistance clinique
    global_str = (f"{train_loss_str}{val_loss_str}"
                  f"Pearson : {metrics['Pearson_pAff']:.4f} | Spearman : {metrics['Spearman_pAff']:.4f} | MAE : {metrics['MAE_pAff']:.4f}")
    
    print(f"\n{global_str}\n")

In [ ]:
import os
import gc
import torch
import torch.optim as optim
from torch.optim.lr_scheduler import OneCycleLR
from torch_geometric.loader import DataLoader

# Nettoyage VRAM
if 'gold_train_loader' in globals(): del globals()['gold_train_loader']
if 'gold_val_loader' in globals(): del globals()['gold_val_loader']
if 'gold_test_loader' in globals(): del globals()['gold_test_loader']
gc.collect()
torch.cuda.empty_cache()

if __name__ == "__main__":
    
    # 1. Paramètres de la phase 2
    BATCH_SIZE = 128
    ACCUM_STEPS = 1  # Lot effectif = 64
    EPOCHS = 30      # 30 époques suffisent amplement grâce à l'initialisation physique
    
    OPTIMAL_WORKERS = min(os.cpu_count(), 8)
    print(f"Déploiement de la puissance brute pour la Phase 2 : {OPTIMAL_WORKERS} workers activés.")

    # 2. Création des DataLoaders Géométriques de Phase 2
    gold_train_loader = DataLoader(
        gold_train_dataset, 
        batch_size=BATCH_SIZE, 
        shuffle=True,
        num_workers=OPTIMAL_WORKERS,
        persistent_workers=True if OPTIMAL_WORKERS > 0 else False,
        pin_memory=True
    )
    
    gold_val_loader = DataLoader(
        gold_val_dataset, 
        batch_size=BATCH_SIZE, 
        shuffle=False, 
        num_workers=0,
        pin_memory=True
    )

    gold_test_loader = DataLoader(
        gold_test_dataset, 
        batch_size=BATCH_SIZE, 
        shuffle=False, 
        num_workers=0,
        pin_memory=True
    )

    # ==========================================
    # 3. Initialisation du manager de perte
    # ==========================================
    loss_manager_p2 = Phase2LossManager(device='cuda')

    # ==========================================
    # 4. Configuration de l'optimiseur : Differential Learning rates
    # ==========================================
    
    new_decay = []
    new_no_decay = []
    
    for name, param in model_phase2.named_parameters():
        if param.requires_grad:  # Si le paramètre est débloqué
            is_no_decay = 'bias' in name or 'norm' in name
            if is_no_decay: 
                new_no_decay.append(param)
            else: 
                new_decay.append(param)
            
    # L'optimiseur ne gère plus que les 18 millions de nouveaux paramètres de Phase 2 
    optimizer_p2 = optim.AdamW([
        {'params': new_decay, 'weight_decay': 0.05},
        {'params': new_no_decay, 'weight_decay': 0.0}
        #{'params': loss_manager_p2.parameters(), 'weight_decay': 0.0}
    ], lr=3e-5)

    total_steps_p2 = len(gold_train_loader) // ACCUM_STEPS * EPOCHS
    scheduler_p2 = OneCycleLR(optimizer_p2, max_lr=3e-5, total_steps=total_steps_p2, pct_start=0.1)
    
    print(f"gold_train_loader.num_workers = {gold_train_loader.num_workers}")
    print(f"L'optimiseur est prêt à entraîner {sum(p.numel() for p in new_decay + new_no_decay):,} paramètres !")

In [ ]:
import torch
import torch.optim as optim
import gc
import os
from tqdm.auto import tqdm

# =============================================================================
# 3. Boucle d'entrainement - Fine-Tuning  (Phase 2)
# =============================================================================
def train_phase2(model, gold_train_loader, gold_val_loader, gold_test_loader, loss_manager, optimizer, scheduler, 
                 epochs=30, device='cuda', accum_steps=2, eval_every_n_steps=100):
    print("\nLancement du Fine-Tuning de la phase 2...")
    
    start_epoch = 0
    global_step = 0
    best_score = -float('inf')
    patience_counter = 0
    patience_limit = 14  # Patience plus courte car le fine-tuning converge rapidement
    best_ckpt_path = None
    
    steps_per_epoch = len(gold_train_loader) // accum_steps
    
    try:
        for epoch in range(start_epoch, epochs):
                    
            model.train()
            running_train_loss = 0.0
            running_steps = 0
            
            pbar = tqdm(total=len(gold_train_loader), desc=f"Epoch {epoch+1}/{epochs} [Fine-Tuning]")
            
            for i, batch in enumerate(gold_train_loader):
                batch = batch.to(device)
                
                with torch.amp.autocast('cuda', dtype=torch.bfloat16):
                    # Forward pass bidirectionnel symétrique sur AEGIS
                    outputs, _, l_w = model(batch)
                    
                    preds_pAff = outputs['delta_pAff']
                    targets_pAff = batch.y_delta_pAff.view(-1, 1)
                    
                    # Calcul de la perte multi-tâches de Phase 2 avec poids d'information clinique [1, 5, 6]
                    loss, _ = loss_manager(
                        preds_pAff, targets_pAff, 
                        batch.gene_id, batch.weight
                    )
                    loss = loss / accum_steps
                    
                loss.backward()
                
                # Accumulation de gradients
                if (i + 1) % accum_steps == 0 or (i + 1) == len(gold_train_loader):
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                    
                    optimizer.step()
                    optimizer.zero_grad()
                    scheduler.step()
                    
                    global_step += 1
                    running_train_loss += (loss.item() * accum_steps)
                    running_steps += 1
                    
                    # =================================================================
                    # Validation intra-époque (Toutes les 100 étapes)
                    # =================================================================
                    if global_step % eval_every_n_steps == 0:
                        avg_train_loss = running_train_loss / max(1, running_steps)
                        
                        # Validation spécifique Phase 2 (max 500 batchs pour aller vite)
                        metrics = validate_phase2(model, gold_val_loader, loss_manager, device, max_batches=500)
                        metrics['Train_Loss'] = avg_train_loss
                        
                        print_detailed_metrics_phase2(metrics)
                        
                        running_train_loss = 0.0
                        running_steps = 0
                        
                        current_score = metrics['Spearman_pAff']
                        
                        # Critère académique strict de minimisation de la perte [5, 6]
                        if current_score > best_score:
                            best_score = current_score
                            patience_counter = 0
                            
                            checkpoint = {
                                'epoch': epoch + 1,
                                'step': global_step,
                                'model_state': model.state_dict(),
                                'optimizer_state': optimizer.state_dict(),
                                'scheduler_state': scheduler.state_dict(),
                                'best_score': best_score
                            }
                            
                            new_ckpt_path = f"aegis_phase2_best_ep{epoch+1}_step{global_step}_score{best_score:.4f}.pth"
                            
                            # Nettoyage de l'ancien fichier de Phase 2
                            if best_ckpt_path is not None and os.path.exists(best_ckpt_path):
                                if "input" not in best_ckpt_path:
                                    os.remove(best_ckpt_path)
                                    
                            torch.save(checkpoint, new_ckpt_path)
                            best_ckpt_path = new_ckpt_path
                            print(f"Nouveau Record de classement (Score : {best_score:.4f}). Checkpoint sauvegardé ({new_ckpt_path}) !")
                        else:
                            patience_counter += 1
                            print(f"Patience: {patience_counter}/{patience_limit}")
                            
                        if patience_counter >= patience_limit:
                            print(f"Early stopping déclenché au step {global_step}.")
                            break
                            
                        model.train()
                        
                pbar.update(1)
                pbar.set_postfix({'Loss': f"{loss.item() * accum_steps:.4f}"})
                
                if (i + 1) >= len(gold_train_loader):
                    break
                    
            pbar.close()
            torch.cuda.empty_cache()
            gc.collect()
            
            if patience_counter >= patience_limit:
                break
                
    except KeyboardInterrupt:
        print(f"\n\nBouton stop pressé ! Sauvegarde d'urgence au step {global_step}...")
        checkpoint_urgence = {
            'epoch': epoch + 1,
            'step': global_step,
            'model_state': model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'scheduler_state': scheduler.state_dict(),
            'best_score': best_score
        }
        nom_fichier = f"aegis_phase2_reprise_step{global_step}.pth"
        torch.save(checkpoint_urgence, nom_fichier)
        print(f"Sauvegarde d'urgence réussie : {nom_fichier}")
        return model
        
    print("Chargement des meilleurs poids pour l'évaluation finale sur le test set...")
    try:
        checkpoint = torch.load(best_ckpt_path, map_location=device, weights_only=False)
        model.load_state_dict(checkpoint['model_state'])
        print(f"Meilleurs poids restaurés depuis : {best_ckpt_path}")
    except Exception as e:
        print(f"Restauration échouée, utilisation des poids actuels. Erreur: {e}")
        
    final_metrics = validate_phase2(model, gold_test_loader, loss_manager, device)
    print_detailed_metrics_phase2(final_metrics)
    
    return model

In [ ]:
train_phase2(
    model=model_phase2, 
    gold_train_loader=gold_train_loader, 
    gold_val_loader=gold_val_loader, 
    gold_test_loader=gold_test_loader, 
    loss_manager=loss_manager_p2, 
    optimizer=optimizer_p2, 
    scheduler=scheduler_p2, 
    epochs=30,  # 30 époques suffisent amplement
    device='cuda', 
    accum_steps=2, 
    eval_every_n_steps=121
)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import numpy as np
 
# ── Données exactes des logs Phase 2 ───────────────────────────────────────────
# Chaque ligne = une évaluation à la fin de chaque époque
# ep5/step121  → première évaluation = meilleur checkpoint
# ep10/step242 → époque 2, patience 1/14
# etc.
 
evals       = [1,       2,       3,       4,       5,       6,       7]
# Labels affichés sur l'axe X
eval_labels = ["ep5\nstep121", "ep10\nstep242", "ep15\nstep363",
               "ep20\nstep484", "ep25\nstep605", "ep30\nstep726", "ep35\nstep847"]
 
train_loss  = [12.3728,  6.2102,  2.7147,  2.1014,  1.4227,  1.4907,  1.4477]
val_loss    = [12.0195,  4.0208,  2.5247,  2.0611,  2.1550,  2.0220,  1.9392]
pearson     = [ 0.0867, -0.0574, -0.0268,  0.0236, -0.0176,  0.0012,  0.0237]
spearman    = [ 0.1189, -0.0676, -0.0198,  0.0418, -0.0082,  0.0163,  0.0427]
mae         = [ 1.5141,  0.6213,  0.1934,  0.1563,  0.1439,  0.1370,  0.1392]
 
# ── Meilleur checkpoint = évaluation 1 (ep5/step121) ──────────────────────────
best_idx       = 0        # index dans les listes ci-dessus
best_label     = "ep5 / step 121"
best_pearson   = pearson[best_idx]    # 0.0867
best_spearman  = spearman[best_idx]   # 0.1189
best_mae       = mae[best_idx]        # 1.5141
best_val_loss  = val_loss[best_idx]   # 12.0195
 
# ── Résultats finaux sur TEST set après restauration du best checkpoint ─────────
test_pearson   = 0.3094
test_spearman  = 0.1317
test_mae       = 1.5187
 
# ── Style ──────────────────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.facecolor": "#0F0F1A",
    "axes.facecolor":   "#0F0F1A",
    "axes.edgecolor":   "#2A2A40",
    "axes.labelcolor":  "#C8C8E0",
    "xtick.color":      "#8888AA",
    "ytick.color":      "#8888AA",
    "text.color":       "#E0E0F0",
    "grid.color":       "#1E1E30",
    "grid.linewidth":   0.6,
    "grid.linestyle":   "--",
    "font.family":      "DejaVu Sans",
    "font.size":        11,
})
 
PURPLE = "#7B68EE"
TEAL   = "#2ECC71"
CORAL  = "#FF6B6B"
AMBER  = "#F0A500"
CYAN   = "#00BFFF"
GRAY   = "#555577"
 
# ── Layout 2×3 ─────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(20, 11))
fig.patch.set_facecolor("#0F0F1A")
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.50, wspace=0.35,
                       width_ratios=[1.2, 1.2, 0.9])
 
ax1 = fig.add_subplot(gs[0, 0])   # Train/Val Loss
ax2 = fig.add_subplot(gs[0, 1])   # Pearson & Spearman
ax3 = fig.add_subplot(gs[0, 2])   # MAE
ax4 = fig.add_subplot(gs[1, 0])   # Bilan corrélations
ax5 = fig.add_subplot(gs[1, 1])   # Bilan MAE
ax6 = fig.add_subplot(gs[1, 2])   # Résumé textuel
 
fig.suptitle(
    "AEGIS-GT — Courbes d'entraînement Phase 2  (Fine-Tuning Pharmacologique)",
    fontsize=14, fontweight="bold", color="#E0E0F0", y=0.99
)
 
x = np.array(evals)
 
# ────────────────────────────────────────────────────────────────────────────
# Panneau 1 — Train Loss & Val Loss
# ────────────────────────────────────────────────────────────────────────────
ax1.set_facecolor("#0F0F1A")
ax1.grid(True)
 
ax1.plot(x, val_loss,   color=PURPLE, lw=2.2, marker="o", ms=6,
         label="Validation Loss", zorder=3)
ax1.plot(x, train_loss, color=TEAL,   lw=2.0, marker="s", ms=5,
         linestyle="--", label="Training Loss", zorder=3)
 
# Zone convergence rapide
ax1.axvspan(1, 3, alpha=0.06, color=TEAL)
ax1.text(1.05, 11.2, "Convergence\nrapide", color=TEAL, fontsize=8, alpha=0.9, va="top")
 
# Zone patience
ax1.axvspan(2, 7.3, alpha=0.04, color=CORAL)
ax1.text(5.05, 9.5, "Patience\n1–6/14", color=CORAL, fontsize=8, alpha=0.9)
 
# Best checkpoint = éval 1 (ep5/step121)
ax1.axvline(x=1, color=AMBER, lw=1.2, linestyle=":", alpha=0.85)
ax1.scatter([1], [best_val_loss], color=AMBER, s=120, zorder=6,
            label=f"Best {best_label}")
ax1.annotate(
    f" Best\n {best_label}\n ρ={best_spearman}",
    xy=(1, best_val_loss),
    xytext=(1.5, best_val_loss - 2.5),
    color=AMBER, fontsize=8.5,
    arrowprops=dict(arrowstyle="->", color=AMBER, lw=1.0)
)
 
ax1.set_title("Pertes d'entraînement & validation",
              color="#C8C8E0", fontsize=12, pad=8)
ax1.set_xlabel("Évaluation")
ax1.set_ylabel("Loss  (Huber + Ranking)")
ax1.set_xticks(x)
ax1.set_xticklabels(eval_labels, fontsize=7.5)
ax1.legend(fontsize=9, framealpha=0.2, loc="upper right",
           facecolor="#1A1A2E", edgecolor="#333355", labelcolor="#C8C8E0")
 
# ────────────────────────────────────────────────────────────────────────────
# Panneau 2 — Pearson & Spearman
# ────────────────────────────────────────────────────────────────────────────
ax2.set_facecolor("#0F0F1A")
ax2.grid(True)
 
ax2.plot(x, pearson,  color=PURPLE, lw=2.2, marker="o", ms=6,
         label="Pearson r  (val)")
ax2.plot(x, spearman, color=CYAN,   lw=2.0, marker="D", ms=5,
         linestyle="--", label="Spearman ρ  (val)")
ax2.axhline(y=0, color=GRAY, lw=1.0, linestyle=":", alpha=0.8,
            label="Aléatoire  (ρ=0)")
 
# Best checkpoint = éval 1
ax2.axvline(x=1, color=AMBER, lw=1.2, linestyle=":", alpha=0.85)
ax2.scatter([1], [best_spearman], color=AMBER, s=120, zorder=6,
            label=f"Best {best_label}  ρ={best_spearman}")
ax2.annotate(
    f" ρ={best_spearman}\n r={best_pearson}",
    xy=(1, best_spearman),
    xytext=(1.6, best_spearman + 0.04),
    color=AMBER, fontsize=8.5,
    arrowprops=dict(arrowstyle="->", color=AMBER, lw=1.0)
)
 
# Lignes test set
ax2.axhline(y=test_pearson,  color=PURPLE, lw=1.1, linestyle="-.", alpha=0.6)
ax2.axhline(y=test_spearman, color=CYAN,   lw=1.1, linestyle="-.", alpha=0.6)
ax2.text(7.15, test_pearson,  f" r={test_pearson}",
         color=PURPLE, fontsize=8, va="center", clip_on=False)
ax2.text(7.15, test_spearman + 0.018, f" ρ={test_spearman}",
         color=CYAN, fontsize=8, va="center", clip_on=False)
ax2.text(7.15, test_pearson + 0.025, "← Test OOD",
         color="#8888AA", fontsize=7, va="center", clip_on=False)
 
ax2.set_title("Corrélations Pearson & Spearman  (val set)",
              color="#C8C8E0", fontsize=12, pad=8)
ax2.set_xlabel("Évaluation")
ax2.set_ylabel("Corrélation")
ax2.set_xticks(x)
ax2.set_xticklabels(eval_labels, fontsize=7.5)
ax2.set_xlim(0.5, 7.5)
ax2.set_ylim(-0.13, 0.40)
ax2.legend(fontsize=8.5, framealpha=0.2, loc="lower right",
           facecolor="#1A1A2E", edgecolor="#333355", labelcolor="#C8C8E0")
 
# ────────────────────────────────────────────────────────────────────────────
# Panneau 3 — MAE
# ────────────────────────────────────────────────────────────────────────────
ax3.set_facecolor("#0F0F1A")
ax3.grid(True)
 
ax3.plot(x, mae, color=CORAL, lw=2.2, marker="o", ms=6,
         label="MAE  (val set)")
ax3.fill_between(x, mae, alpha=0.10, color=CORAL)
 
ax3.axvline(x=1, color=AMBER, lw=1.2, linestyle=":", alpha=0.85)
ax3.scatter([1], [best_mae], color=AMBER, s=120, zorder=6,
            label=f"Best {best_label}  {best_mae:.4f}")
ax3.annotate(
    f" MAE={best_mae:.4f}",
    xy=(1, best_mae),
    xytext=(1.5, best_mae - 0.15),
    color=AMBER, fontsize=8.5,
    arrowprops=dict(arrowstyle="->", color=AMBER, lw=1.0)
)
 
ax3.set_title("MAE — delta_pAff  (val set)",
              color="#C8C8E0", fontsize=12, pad=8)
ax3.set_xlabel("Évaluation")
ax3.set_ylabel("MAE  (unités pAff)")
ax3.set_xticks(x)
ax3.set_xticklabels(eval_labels, fontsize=7.5)
ax3.legend(fontsize=9, framealpha=0.2,
           facecolor="#1A1A2E", edgecolor="#333355", labelcolor="#C8C8E0")
 
# ────────────────────────────────────────────────────────────────────────────
# Panneau 4 — Bilan corrélations val best vs test OOD
# ────────────────────────────────────────────────────────────────────────────
ax4.set_facecolor("#0F0F1A")
ax4.grid(True, axis="x")
 
corr_labels = ["Pearson r", "Spearman ρ"]
val_corr    = [best_pearson,  best_spearman]
test_corr   = [test_pearson,  test_spearman]
 
y_pos  = np.arange(len(corr_labels))
height = 0.30
 
b1 = ax4.barh(y_pos + height/2, val_corr,  height=height,
              color=PURPLE, alpha=0.85,
              label=f"Val best ({best_label})", edgecolor="none")
b2 = ax4.barh(y_pos - height/2, test_corr, height=height,
              color=AMBER,  alpha=0.85,
              label="Test set Gene-Disjoint OOD", edgecolor="none")
 
ax4.set_yticks(y_pos)
ax4.set_yticklabels(corr_labels, fontsize=11, color="#C8C8E0")
ax4.axvline(x=0, color=GRAY, lw=0.8)
ax4.set_xlabel("Corrélation")
ax4.set_xlim(-0.10, 0.45)
ax4.set_title(f"Bilan corrélations — Val best vs Test OOD",
              color="#C8C8E0", fontsize=12, pad=8)
 
for bar, val in zip(b1, val_corr):
    offset = 0.008 if val >= 0 else -0.008
    ha     = "left" if val >= 0 else "right"
    ax4.text(val + offset, bar.get_y() + bar.get_height()/2,
             f"{val:.4f}", va="center", ha=ha, fontsize=10, color="#E0E0F0")
 
for bar, val in zip(b2, test_corr):
    offset = 0.008 if val >= 0 else -0.008
    ha     = "left" if val >= 0 else "right"
    ax4.text(val + offset, bar.get_y() + bar.get_height()/2,
             f"{val:.4f}", va="center", ha=ha, fontsize=10, color="#E0E0F0")
 
ax4.legend(fontsize=9, framealpha=0.2, loc="lower right",
           facecolor="#1A1A2E", edgecolor="#333355", labelcolor="#C8C8E0")
 
ax4.text(0.03, 0.08,
    f"Pearson test (0.31) >> val ({best_pearson:.4f})\n"
    "→ Généralisation OOD confirmée\n"
    "Test > val : pas d'overfitting",
    transform=ax4.transAxes, fontsize=8, color="#AAAACC", va="bottom",
    bbox=dict(boxstyle="round,pad=0.4", facecolor="#1A1A2E",
              edgecolor=TEAL, alpha=0.85, linewidth=1.2))
 
# ────────────────────────────────────────────────────────────────────────────
# Panneau 5 — Bilan MAE séparé
# ────────────────────────────────────────────────────────────────────────────
ax5.set_facecolor("#0F0F1A")
ax5.grid(True, axis="x")
 
mae_labels = [f"Val best\n({best_label})", "Test set OOD"]
mae_values = [best_mae, test_mae]
mae_colors = [PURPLE, AMBER]
 
y_mae  = np.arange(len(mae_labels))
b_mae  = ax5.barh(y_mae, mae_values, height=0.40,
                  color=mae_colors, alpha=0.85, edgecolor="none")
 
ax5.set_yticks(y_mae)
ax5.set_yticklabels(mae_labels, fontsize=10, color="#C8C8E0")
ax5.set_xlabel("MAE  (unités pAff)")
ax5.set_title("Bilan MAE — Val best vs Test OOD",
              color="#C8C8E0", fontsize=12, pad=8)
ax5.set_xlim(0, 1.85)
 
for bar, val in zip(b_mae, mae_values):
    ax5.text(val + 0.03, bar.get_y() + bar.get_height()/2,
             f"{val:.4f}", va="center", ha="left",
             fontsize=10, color="#E0E0F0")
 
ax5.text(0.03, 0.12,
    f"MAE val ({best_mae:.4f}) ≈ MAE test ({test_mae})\n"
    "→ Cohérence val/test sur la MAE\n"
    "Erreur ~1.5 pAff = facteur ~33×\n"
    "sur l'affinité absolue",
    transform=ax5.transAxes, fontsize=8, color="#AAAACC", va="bottom",
    bbox=dict(boxstyle="round,pad=0.4", facecolor="#1A1A2E",
              edgecolor=AMBER, alpha=0.85, linewidth=1.2))
 
# ────────────────────────────────────────────────────────────────────────────
# Panneau 6 — Résumé textuel
# ────────────────────────────────────────────────────────────────────────────
ax6.set_facecolor("#0F0F1A")
ax6.axis("off")
 
lines = [
    ("RÉSUMÉ PHASE 2",              "#E0E0F0", 13, True),
    ("",                            "#E0E0F0",  8, False),
    ("Meilleur checkpoint",         "#8888AA",  9, False),
    (f"  {best_label}",             AMBER,     11, True),
    ("",                            "#E0E0F0",  7, False),
    ("Val set (best checkpoint)",   "#8888AA",  9, False),
    (f"  Pearson r  = {best_pearson:.4f}", PURPLE, 11, True),
    (f"  Spearman ρ = {best_spearman}",   CYAN,   11, True),
    (f"  MAE        = {best_mae}",         CORAL,  11, True),
    (f"  Val Loss   = {best_val_loss}",    GRAY,   10, False),
    ("",                            "#E0E0F0",  7, False),
    ("Test set Gene-Disjoint OOD",  "#8888AA",  9, False),
    (f"  Pearson r  = {test_pearson}", PURPLE, 11, True),
    (f"  Spearman ρ = {test_spearman}", CYAN,   11, True),
    (f"  MAE        = {test_mae}",     CORAL,  11, True),
    ("",                            "#E0E0F0",  7, False),
    ("Protocole",                   "#8888AA",  9, False),
    ("  Gene-Disjoint OOD",         "#C8C8E0", 10, False),
    ("  Aucun gène du test set",    "#C8C8E0", 10, False),
    ("  présent à l'entraînement",  "#C8C8E0", 10, False),
    ("",                            "#E0E0F0",  7, False),
    ("Modèle",                      "#8888AA",  9, False),
    ("  AEGIS-GT  22.9M params",    "#C8C8E0", 10, False),
    ("  Backbone Phase 1 gelé",     "#C8C8E0", 10, False),
    ("  Fine-tuning sélectif",      "#C8C8E0", 10, False),
]
 
y_txt = 0.97
for text, color, size, bold in lines:
    weight = "bold" if bold else "normal"
    ax6.text(0.05, y_txt, text, transform=ax6.transAxes,
             fontsize=size, color=color, fontweight=weight,
             va="top", ha="left")
    y_txt -= 0.042 if size >= 11 else 0.034
 
for spine in ["top","bottom","left","right"]:
    ax6.spines[spine].set_visible(True)
    ax6.spines[spine].set_edgecolor("#2A2A40")
    ax6.spines[spine].set_linewidth(0.8)
 
# ── Bandeau bas ───────────────────────────────────────────────────────────────
fig.text(
    0.5, 0.005,
    f"Best : {best_label}  |  "
    f"Val r={best_pearson}  Val ρ={best_spearman}  Val MAE={best_mae}  |  "
    f"Test r={test_pearson}  Test ρ={test_spearman}  Test MAE={test_mae}  |  "
    "Protocole : Gene-Disjoint OOD  |  AEGIS-GT 22.9M params",
    ha="center", va="bottom", fontsize=8.5, color="#7777AA",
    bbox=dict(boxstyle="round,pad=0.4", facecolor="#111122",
              edgecolor="#2A2A40", alpha=0.85)
)
 
plt.savefig("aegis_phase2_training_curves.png",
            dpi=180, bbox_inches="tight",
            facecolor="#0F0F1A", edgecolor="none")
plt.show()

In [ ]:
from IPython.display import FileLink
import os

# Vérification de l'existence du fichier avant de générer le lien
file_name = 'aegis_phase2_best_ep5_step121_score0.1189.pth'

if os.path.exists(file_name):
    print(f"Le fichier '{file_name}' est prêt pour le téléchargement.")
    # Génère le lien bleu cliquable
    display(FileLink(file_name))
else:
    print(f"Erreur : Le fichier '{file_name}' n'a pas été trouvé dans /kaggle/working/")


In [ ]:
# Cellule d'audit de dérive des poids 
def audit_weight_drift(checkpoint_phase1_path, model_phase2):
    print("Audit de Dérive des Poids d'AEGIS-GT :")
    
    # Chargement des poids d'origine de la Phase 1
    checkpoint_p1 = torch.load(checkpoint_phase1_path, map_location='cpu', weights_only=False)
    state_dict_p1 = checkpoint_p1['model_state']
    state_dict_p2 = model_phase2.state_dict()
    
    # on sélectionne quelques couches clés du backbone géométrique à auditer
    backbone_layers_to_audit = [
        'p_proj.weight',
        'layers.0.qkv_p.weight',
        'layers.1.out_p.weight',
        'layers.2.ff.w3.weight'
    ]
    
    for name in backbone_layers_to_audit:
        if name in state_dict_p1 and name in state_dict_p2:
            w_p1 = state_dict_p1[name].float()
            w_p2 = state_dict_p2[name].float().cpu()
            
            # Calcul de la déviation relative L2 : ||W_phase2 - W_phase1|| / ||W_phase1|| [4]
            drift = torch.norm(w_p2 - w_p1) / torch.norm(w_p1)
            print(f"Couche [{name:<25}] : Dérive relative L2 = {drift.item():.4%}")

# Appel de l'audit (remplacez par le nom exact de votre checkpoint Phase 1)
audit_weight_drift("/kaggle/input/datasets/anisis/bgt-checkpoint/aegis_best_ep41_step2700_mae1.0748.pth", model_phase2)